# More Than a Hack 2027 — FINAL possession model, v4 (reproducible)

**Team: Cules of Duty**

**Goal:** one label (`H` / `A` / `D` / `X`) for every one of the 199,319 frames of the tactical video. Ranking metric: spellF1 (possessions matched by temporal IoU), tie-break macroF1.

What this notebook does:

* **Three emission designs** (hierarchical / balanced / mixed) turn two calibrated models (home vs away, dead vs alive) into per-frame log-probabilities that a Viterbi decoder uses, with a minimum run length.
* **A team-model comparison.** Six home/away models are compared on the labelled window (out-of-fold AUC and accuracy), and the best is used.
* **A learned dead-ball model** (Extra Trees on the ball cues, cross-fitted leave-one-minute-out and Platt-calibrated) is compared with two linear dead-ball models.
* **Honest selection.** Settings are chosen *nested* (four minutes of the window to choose, the fifth to score, for every minute). Candidates are ranked by `0.5 × spellF1 + 0.5 × nested Accuracy` among those whose whole-match structure is sane and whose window and whole-match dead operating points agree.
* **Reproducibility.** The two learned estimators are pinned in `PIN_ESTIMATORS`; no model search runs on the machine that reproduces the file. `requirements-pinned.txt` is written at the end.

Everything is fitted from the tracking, the Smart Tagging data and the five published ground-truth minutes. `FINAL_DEAD_TARGET = 0.38` is an assumption from general football knowledge (the window itself is 20% dead).

**Reminders:** the labelled window is the first five minutes only. Keep the match data and `cache/` out of git. Run `python reproduce.py --notebook possession_pipeline_FINAL_v4.ipynb` twice and compare the MD5.

In [1]:
import os, sys, json, time, hashlib, warnings, copy
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find the repository root (the folder that contains games.py), wherever the notebook was opened.
ROOT = Path.cwd().resolve()
while not (ROOT / "games.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("games.py not found - open this notebook inside the challenge repository.")
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "evaluation", ROOT / "starter"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import games                      # per-game constants, asset paths, half boundaries
import possession_metrics as pm   # the official scorer (the file is the specification)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 100
print("repo root:", ROOT)
print("numpy", np.__version__, "| pandas", pd.__version__)

repo root: /Users/patrickkingori/Downloads/more-than-a-hack-2027-main
numpy 1.26.4 | pandas 2.1.4


In [2]:
TEAM_NAME = "Cules of Duty"  # -> evaluation/results/Cules of Duty.csv  (the form wants <team_name>.csv / .pdf; paths contain spaces: quote them in a shell)
FORCE_SYNTHETIC = False      # True = ignore the real data and run the dry-run mini match

CFG = dict(
    # ---- Step 4: ball cleaning (feature parameters)
    max_gap_s=0.6,            # interpolate ball gaps up to this long (seconds)
    max_ball_speed=45.0,      # m/s; isolated jumps faster than this are tracking glitches
    smooth_frames=3,          # box-filter width for ball position (frames); 1 = off
    ball_method="auto",       # "linear" | "kalman" | "auto" (auto = whichever recovers masked real observations better, Step 4b)
    kf_accel_std=12.0,        # Kalman: ball acceleration noise (m/s^2). Larger = follows kicks, smooths less
    kf_meas_std=0.4,          # Kalman: tracking noise of one ball observation (m)
    kf_gate=8.0,              # Kalman: reject observations further than this many sigma from the prediction
    kf_max_gap_s=1.0,         # Kalman: fill gaps up to this long (seconds); longer gaps stay missing
    kf_reacquire_frames=15,   # Kalman: after this many consecutive rejections restart the track on the new observation
    kf_replace_observed=False,# True = also replace observed positions by their smoothed value
    kf_std_cap=3.0,           # filled frames count less as their position std (m) approaches this
    use_video_ball=False,     # Step 7c: add OpenCV ball candidates (needs the video) as extra Kalman measurements
    decoder="viterbi",        # "viterbi" (min-duration clean-up) | "hsmm" (explicit duration model, Step 6d)
    hsmm_step=5,              # HSMM decodes at fps/hsmm_step to stay fast
    hsmm_max_dur_s=60.0,      # longest single H/A/D run the HSMM will consider
    dur_alive_median_s=8.0, dur_alive_sigma=0.9,    # log-normal duration prior of a possession (seconds)
    dur_dead_median_s=8.0, dur_dead_sigma=0.7,      # ... and of a dead-ball stretch
    w_dur=1.0,                # weight of the duration prior

    # ---- Step 5: evidence (feature parameters)
    sigma_m=3.0,              # metres; width of a player's "control" kernel around the ball
    evidence_clip=4.0,        # cap on |log control ratio|
    min_control=0.02,         # below this total control mass the ball is "loose": no team evidence
    pitch_margin_m=0.5,       # ball this far outside the pitch rectangle counts as out of play
    static_speed=0.8,         # m/s; ball slower than this ...
    static_window_s=1.0,      # ... for this whole window counts as "static"
    missing_min_s=3.0,        # ball missing this long counts as weak dead-ball evidence

    # ---- Step 6: emission weights (nats per frame)
    w_ctrl=0.10,              # weight of the control evidence (H positive, A negative)
    w_st_team=0.15,           # weight of the Smart Tagging team prior (weak: it is inferred, not truth)
    interp_weight=0.5,        # evidence from interpolated ball frames counts this much
    dead_bias=0.6,            # per-frame cost of being in D (alive play is the default)
    w_out=1.5,                # ball outside the pitch
    w_static=0.6,             # ball static
    w_setpiece=0.8,           # inside a Smart Tagging set-piece interval
    w_missing=0.10,           # ball missing for a long time
    dead_cues={},             # {feature name from ML_COLS: weight per std}: extra dead-ball cues, chosen with Step 6e (empty = none)

    # ---- Step 6: transition costs (nats, charged once per switch)
    switch_cost=15.0,         # H <-> A   (raise = fewer, longer possessions)
    dead_cost=10.0,           # alive <-> D

    # ---- Step 6: post-processing (seconds)
    min_possession_s=1.0,     # alive runs shorter than this are absorbed by a neighbour
    min_dead_s=2.0,           # dead runs shorter than this are absorbed by a neighbour

    # ---- Smart Tagging interpretation (verify in Step 7 diagnostics!)
    st_home_keys=("barc",),                 # substring of the Team tag that means home
    st_away_keys=("atl", "ath"),            # ... and away
    st_flip_prefixes=("DEFENDING", "DEFENSIVE TRANSITION"),  # Team tag = the DEFENDING side -> flip sign
    st_skip_codes=("SET PIECES",),          # codes that carry no possession-team information
)
print(json.dumps({k: v for k, v in CFG.items()}, indent=1, default=list)[:400], "...")

{
 "max_gap_s": 0.6,
 "max_ball_speed": 45.0,
 "smooth_frames": 3,
 "ball_method": "auto",
 "kf_accel_std": 12.0,
 "kf_meas_std": 0.4,
 "kf_gate": 8.0,
 "kf_max_gap_s": 1.0,
 "kf_reacquire_frames": 15,
 "kf_replace_observed": false,
 "kf_std_cap": 3.0,
 "use_video_ball": false,
 "decoder": "viterbi",
 "hsmm_step": 5,
 "hsmm_max_dur_s": 60.0,
 "dur_alive_median_s": 8.0,
 "dur_alive_sigma": 0.9,
 "d ...


### Dry-run data generator (only used when the real data is not found)

In [3]:
def make_synthetic_match(out_dir, seed=0, n_frames=24000, fps=25, pitch=(105.3, 68.0)):
    """Fake match in the real file formats. NOT football; only enough structure to test the code."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    L, W = pitch
    halves = [(1, 500, 11999), (2, 12500, 23499)]
    kick2 = 12700                                     # 2nd half is dead until here (like the real 113742)

    # 1) possession timeline
    state = np.full(n_frames, "X", dtype="<U1")
    for half, a, b in halves:
        t, team = a, str(rng.choice(["H", "A"]))
        if half == 2:
            state[a:kick2] = "D"; t = kick2
        while t <= b:
            dur = int(np.clip(rng.lognormal(np.log(9 * fps), 0.7), 2 * fps, 40 * fps))
            end = min(t + dur, b + 1); state[t:end] = team; t = end
            if t > b:
                break
            if rng.random() < 0.45:                  # stoppage, then a restart by a random team
                end = min(t + int(rng.uniform(4, 14) * fps), b + 1); state[t:end] = "D"; t = end
                team = str(rng.choice(["H", "A"]))
            else:                                    # direct turnover
                team = "A" if team == "H" else "H"

    # 2) players and ball
    base = np.zeros((22, 2))
    for i in range(11):
        base[i] = [rng.uniform(15, 60), rng.uniform(8, 60)]
        base[11 + i] = [rng.uniform(45, 90), rng.uniform(8, 60)]
    pos, ball_pos, carrier, last_team = base.copy(), np.array([L / 2, W / 2]), 0, None
    P = np.full((n_frames, 22, 2), np.nan); B = np.full((n_frames, 2), np.nan)
    for t in range(n_frames):
        s = state[t]
        if s in ("H", "A"):
            team = 0 if s == "H" else 1
            if team != last_team or rng.random() < 1 / 75:
                idx = np.arange(11) + 11 * team
                d = np.hypot(*(pos[idx] - ball_pos).T)
                carrier = idx[rng.choice(np.argsort(d)[:4])]
            last_team = team
            delta = pos[carrier] + rng.normal(0, 0.4, 2) - ball_pos
            ball_pos = ball_pos + delta * min(0.15, 0.7 / max(float(np.hypot(*delta)), 1e-6))   # <= 0.7 m per frame (~17 m/s)
        elif s == "D":
            if t == 0 or state[t - 1] != "D":
                kind = rng.integers(3)
                ball_pos = np.array([rng.uniform(5, L - 5), -0.2 if rng.random() < .5 else W + 0.2]) if kind == 0 \
                    else np.array([rng.choice([5.5, L - 5.5]), rng.uniform(20, 48)]) if kind == 1 \
                    else np.array([rng.uniform(10, L - 10), rng.uniform(10, W - 10)])
            last_team = None
        pos = pos + 0.02 * ((base + 0.3 * (ball_pos - base)) - pos) + rng.normal(0, 0.06, pos.shape)
        if s != "X":
            P[t] = pos; B[t] = ball_pos
    P += rng.normal(0, 0.25, P.shape)
    B += rng.normal(0, 1, B.shape) * np.where(state == "D", 0.03, 0.25)[:, None]   # a resting ball jitters less

    def dropout_mask(n, mean_vis, mean_miss):
        m, t, vis = np.zeros(n, bool), 0, True
        while t < n:
            k = int(rng.geometric(1 / (mean_vis if vis else mean_miss)))
            m[t:t + k] = not vis; t += k; vis = not vis
        return m
    B_full, P_full = B.copy(), P.copy()                              # before the tactical dropouts (the broadcast has its own)
    B[dropout_mask(n_frames, 60, 40)] = np.nan                      # ~40% ball dropouts
    for i in range(22):
        P[dropout_mask(n_frames, 400, 25), i] = np.nan               # player dropouts
    V = np.zeros((n_frames, 22)); V[1:] = np.hypot(*np.diff(P, axis=0).transpose(2, 0, 1)) * fps
    V = np.nan_to_num(V)

    # 3) write the ATD in the real raw format: "<frame_id>:<x,y,s;...46 slots...>:<ball>"
    slot_of = list(range(11)) + list(range(23, 34))
    with open(out_dir / "SYN_atd.txt", "w") as fh:
        for t in range(n_frames):
            groups = ["NaN,NaN,NaN"] * 46
            for i in range(22):
                if not np.isnan(P[t, i, 0]):
                    groups[slot_of[i]] = "%.6f,%.6f,%.3f" % (P[t, i, 0] / L, P[t, i, 1] / W, V[t, i])
            bs = "NaN,NaN,NaN" if np.isnan(B[t, 0]) else "%.6f,%.6f,0.000" % (B[t, 0] / L, B[t, 1] / W)
            fh.write("%d:%s:%s\n" % (t + 1, ";".join(groups), bs))
    players = "".join('<Player id="Track %d" teamId="%s"><ShirtNumber>%d</ShirtNumber></Player>'
                      % (i + 1, "HOME" if i < 23 else "AWAY", i + 1) for i in range(46))
    (out_dir / "SYN_atd.xml").write_text(
        '<?xml version="1.0"?><main><Metadata><Teams><Team id="HOME"><Name>Barcelona</Name></Team>'
        '<Team id="AWAY"><Name>Athletic Club</Name></Team></Teams><Players>%s</Players></Metadata></main>' % players)

    # 4) Smart Tagging: noisy phase tags per possession, set-piece tags per stoppage
    names = {"H": "Barcelona", "A": "Atletic Club"}
    inst, run_start = [], 0
    for t in range(1, n_frames + 1):
        if t == n_frames or state[t] != state[run_start]:
            s, a, b = state[run_start], run_start, t
            if s in ("H", "A") and rng.random() < 0.8:
                code_ = str(rng.choice(["BUILD UP", "PROGRESSION", "BALL IN FINAL THIRD", "DEFENDING IN MIDDLE THIRD"]))
                tag = s if not code_.startswith("DEFENDING") else ("A" if s == "H" else "H")
                if rng.random() < 0.15:
                    tag = "A" if tag == "H" else "H"
                inst.append((a / fps, b / fps, code_, "Team", names[tag], ""))
            elif s == "D" and a >= 500 and rng.random() < 0.8:
                typ = str(rng.choice(["Throw-in", "Corner Kick", "Goal Kick", "Free Kick"]))
                inst.append(((a - 50) / fps, (b + 25) / fps, "SET PIECES", "Team",
                             "N/A" if typ == "Throw-in" else names[str(rng.choice(["H", "A"]))], typ))
            run_start = t
    body = "".join("<instance><ID>%d</ID><start>%.2f</start><end>%.2f</end><code>%s</code>"
                   "<label><group>%s</group><text>%s</text></label>%s</instance>"
                   % (i, a, b, c, g, x, ("<label><group>Type</group><text>%s</text></label>" % t) if t else "")
                   for i, (a, b, c, g, x, t) in enumerate(inst))
    (out_dir / "SYN_smart_tagging.xml").write_text("<file><ALL_INSTANCES>%s</ALL_INSTANCES></file>" % body)

    # 4b) a fake BROADCAST source: shifted in time, partial coverage (cuts), only players near the ball, own dropouts
    off, nb_ = 137, n_frames - 400                                   # broadcast frame b shows tactical frame b + off
    cam_on = np.zeros(nb_, bool); t_ = 0
    while t_ < nb_:
        k = int(rng.integers(300, 900)); cam_on[t_:t_ + k] = rng.random() < 0.7; t_ += k
    with open(out_dir / "SYN_broadcast_atd.txt", "w") as fh:
        for b_ in range(nb_):
            groups, ball_s, tt = ["NaN,NaN,NaN"] * 46, "NaN,NaN,NaN", b_ + off
            if cam_on[b_] and not np.isnan(B_full[tt, 0]):
                cx = B_full[tt, 0]
                for i_ in range(22):
                    if not np.isnan(P_full[tt, i_, 0]) and abs(P_full[tt, i_, 0] - cx) < 30:
                        groups[slot_of[i_]] = "%.6f,%.6f,0.000" % (P_full[tt, i_, 0] / L, P_full[tt, i_, 1] / W)
                if rng.random() > 0.25:
                    ball_s = "%.6f,%.6f,0.000" % ((B_full[tt, 0] + rng.normal(0, 0.2)) / L, (B_full[tt, 1] + rng.normal(0, 0.2)) / W)
            fh.write("%d:%s:%s\n" % (b_ + 1, ";".join(groups), ball_s))

    # 5) a fixed homography (normalised pitch 0-1 -> 1920x1080 pixels), same CSV layout as the real one
    Hm = np.array([[1320.0, -256.0, 300.0], [0.0, 533.3, 200.0], [0.0, -0.2667, 1.0]])
    ht = pd.DataFrame({"frame": np.arange(n_frames)})
    for name, v in zip(["h00", "h01", "h02", "h10", "h11", "h12", "h20", "h21", "h22"], Hm.flatten()):
        ht[name] = v
    ht["reprojection_error"] = 0.25; ht["valid"] = 1
    ht.to_csv(out_dir / "SYN_homography.csv", index=False)

    return dict(meta=str(out_dir / "SYN_atd.xml"), raw=str(out_dir / "SYN_atd.txt"),
                smart_tagging=str(out_dir / "SYN_smart_tagging.xml"), n_frames=n_frames, fps=fps,
                pitch=pitch, halves=halves, pre_kickoff_dead={2: kick2}, truth=state,
                homography=str(out_dir / "SYN_homography.csv"),
                broadcast_raw=str(out_dir / "SYN_broadcast_atd.txt"), broadcast_frames=nb_, broadcast_offset=off)

## Load data

In [4]:
SYNTHETIC = FORCE_SYNTHETIC
if not SYNTHETIC:
    try:
        REAL_PATHS = dict(zip(["atd_meta", "atd_raw", "smart_tagging"],
                              games.require(["atd_meta", "atd_raw", "smart_tagging"])))
    except IOError as err:
        print("Real data not found -> DRY-RUN MODE with synthetic data.\nExpected e.g.:", str(err).splitlines()[1].strip())
        SYNTHETIC = True

if SYNTHETIC:
    syn = make_synthetic_match(ROOT / "cache" / "synthetic")
    MATCH = dict(n_frames=syn["n_frames"], fps=syn["fps"], pitch=syn["pitch"], halves=syn["halves"],
                 meta=syn["meta"], raw=syn["raw"], tagging=syn["smart_tagging"],
                 pre_kickoff_dead=syn["pre_kickoff_dead"], homography=syn["homography"], video=None,
                 broadcast_raw=syn["broadcast_raw"], broadcast_frames=syn["broadcast_frames"], broadcast_offset=syn["broadcast_offset"])
    GT_LETTERS = syn["truth"]
    GT_NAME = "SYNTHETIC truth (whole fake match)"
else:
    g = games.GAMES[games.GAME]
    MATCH = dict(n_frames=games.source_frames(), fps=g["fps"], pitch=g["pitch"],
                 halves=games.half_video_blocks(games.GAME),            # [(1, 11185, 90651), (2, 110808, 190197)]
                 meta=REAL_PATHS["atd_meta"], raw=REAL_PATHS["atd_raw"], tagging=REAL_PATHS["smart_tagging"],
                 pre_kickoff_dead={2: 113742})                           # FAQ: 2nd half graded from 110808 but kicks off at 113742; all D before
    for _key in ("homography", "video"):                      # optional assets: only needed for Steps 7b-7d
        try:
            MATCH[_key] = games.require([_key])[0]
        except IOError:
            MATCH[_key] = None
    try:                                                      # broadcast tracking (optional, NOT synchronised to the tactical video)
        _b = games.asset_paths(source="broadcast")
        MATCH["broadcast_raw"] = _b["atd_raw"] if Path(_b["atd_raw"]).exists() else None
        MATCH["broadcast_frames"] = games.source_frames(source="broadcast")
    except Exception:
        MATCH["broadcast_raw"], MATCH["broadcast_frames"] = None, 0
    gt_path = ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"
    GT_LETTERS = pm.read_label_csv(gt_path)["label"].astype(str).str.strip().str.upper().values
    GT_NAME = "PUBLIC ground truth (first 5 min of play, 17 possessions)"

N, FPS = MATCH["n_frames"], MATCH["fps"]
L_PITCH, W_PITCH = MATCH["pitch"]
HALVES = MATCH["halves"]
GT_CODES = pm.label_codes(GT_LETTERS)
GRADED = np.zeros(N, bool)
for _, a, b in HALVES:
    GRADED[a:b + 1] = True
print("mode:", "SYNTHETIC dry-run" if SYNTHETIC else "REAL data")
print("frames:", N, "| fps:", FPS, "| pitch:", MATCH["pitch"])
print("halves (half, first, last):", HALVES, "| graded frames:", int(GRADED.sum()))
print("scoring reference:", GT_NAME, "| labelled frames:", int((GT_CODES != pm.X).sum()))

mode: REAL data
frames: 199319 | fps: 25 | pitch: (105.3, 68.0)
halves (half, first, last): [(1, 11185, 90651), (2, 110808, 190197)] | graded frames: 158857
scoring reference: PUBLIC ground truth (first 5 min of play, 17 possessions) | labelled frames: 7500


In [5]:
def parse_atd(raw_path, n_frames):
    """Stream the raw ATD into arrays. Returns players (N,S,3: x,y,speed) and ball (N,3), normalised 0-1."""
    st = Path(raw_path).stat()
    cache = ROOT / "cache" / ("atd_%d_%d.npz" % (st.st_size, int(st.st_mtime)))
    if cache.exists():
        z = np.load(cache); print("loaded cache", cache.name); return z["players"], z["ball"]
    players = ball = None; n_slots = bad = 0; t0 = time.time()
    with open(raw_path) as fh:
        for line in fh:
            parts = line.rstrip("\r\n").split(":")
            if len(parts) != 3:
                continue
            fid = int(parts[0]) - 1                                  # frame_id counts from 1
            vals = np.fromstring(parts[1].replace(";", ",").strip(","), sep=",", dtype=np.float32)
            if players is None:
                n_slots = len(vals) // 3
                players = np.full((n_frames, n_slots, 3), np.nan, np.float32)
                ball = np.full((n_frames, 3), np.nan, np.float32)
            if not (0 <= fid < n_frames) or len(vals) != n_slots * 3:
                bad += 1; continue
            players[fid] = vals.reshape(n_slots, 3)
            bv = np.fromstring(parts[2].strip().strip(";"), sep=",", dtype=np.float32)[:3]
            ball[fid, :len(bv)] = bv
    print("parsed %d slots in %.0fs (%d malformed lines skipped)" % (n_slots, time.time() - t0, bad))
    cache.parent.mkdir(exist_ok=True)
    np.savez(cache, players=players, ball=ball)
    return players, ball

def strip_ns(tag):
    return tag.split("}")[-1]

def read_teams(meta_path):
    """(home_name, away_name, slot_team) with slot_team[s] = 0 for home, 1 for away."""
    root = ET.parse(meta_path).getroot()
    teams = [e for e in root.iter() if strip_ns(e.tag) == "Team"]
    ids = [t.get("id") for t in teams]
    names = [(next((c.text for c in t if strip_ns(c.tag) == "Name"), None) or t.get("id")) for t in teams]
    assert len(ids) == 2, "expected 2 teams in the metadata, found %d" % len(ids)
    slot_team = np.array([0 if p.get("teamId") == ids[0] else 1
                          for p in root.iter() if strip_ns(p.tag) == "Player"], dtype=np.int8)
    return names[0], names[1], slot_team

def read_smart_tagging(path, fps):
    rows = []
    for inst in ET.parse(path).getroot().iter("instance"):
        s, e = float(inst.findtext("start")), float(inst.findtext("end"))
        tags = {l.findtext("group"): l.findtext("text") for l in inst.findall("label")}
        rows.append(dict(code=(inst.findtext("code") or "").strip().upper(),
                         start=int(round(s * fps)), end=int(round(e * fps)),   # end is exclusive
                         team=(tags.get("Team") or "").strip(), type=(tags.get("Type") or "").strip(),
                         half=(tags.get("Half") or "").strip()))
    return pd.DataFrame(rows)

raw_players, raw_ball = parse_atd(MATCH["raw"], N)
home_name, away_name, slot_team = read_teams(MATCH["meta"])
S = raw_players.shape[1]
assert len(slot_team) == S, "metadata lists %d slots but the ATD has %d" % (len(slot_team), S)

P = np.stack([raw_players[:, :, 0] * L_PITCH, raw_players[:, :, 1] * W_PITCH], axis=2)   # metres, (N,S,2)
PSPD = raw_players[:, :, 2]                                                               # m/s
B = np.stack([raw_ball[:, 0] * L_PITCH, raw_ball[:, 1] * W_PITCH], axis=1)               # metres, (N,2)
del raw_players, raw_ball

ST = read_smart_tagging(MATCH["tagging"], FPS)
print("teams: H = %s | A = %s | slots: %d (home %d / away %d)" % (home_name, away_name, S, (slot_team == 0).sum(), (slot_team == 1).sum()))
print("Smart Tagging instances:", len(ST))
ST.head()

loaded cache atd_195714285_1791353242.npz
teams: H = Barcelona | A = Atletic Club | slots: 46 (home 24 / away 22)
Smart Tagging instances: 638


,code,start,end,team,type,half
0,SET PIECES,11097,11222,N/A,Kick Off (Start),1st Half
1,SET PIECES,49631,49756,N/A,Kick Off (After Goal),1st Half
2,SET PIECES,62012,62137,N/A,Kick Off (After Goal),1st Half
3,SET PIECES,75457,75582,N/A,Kick Off (After Goal),1st Half
4,SET PIECES,113660,113785,N/A,Kick Off (Start),2nd Half


## Sanity checks and alignment check (the baseline must reproduce the published numbers)

In [6]:
def run_lengths(mask):
    """(starts, lengths) of the runs of True in a boolean array."""
    m = np.concatenate(([0], mask.astype(np.int8), [0]))
    d = np.diff(m)
    s, e = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    return s, e - s

ball_obs = ~np.isnan(B[:, 0])
n_home = (~np.isnan(P[:, :, 0]))[:, slot_team == 0].sum(1)
n_away = (~np.isnan(P[:, :, 0]))[:, slot_team == 1].sum(1)
margin = CFG["pitch_margin_m"]
outside = ball_obs & ((B[:, 0] < -margin) | (B[:, 0] > L_PITCH + margin) | (B[:, 1] < -margin) | (B[:, 1] > W_PITCH + margin))

print("ball visible: all frames %.1f%% | graded frames %.1f%%" % (100 * ball_obs.mean(), 100 * ball_obs[GRADED].mean()))
print("tracked players per frame (graded): home %.1f, away %.1f" % (n_home[GRADED].mean(), n_away[GRADED].mean()))
print("ball observed OUTSIDE the pitch (+%.1f m) on %.2f%% of graded frames" % (margin, 100 * outside[GRADED].mean()))
gs, gl = run_lengths(~ball_obs & GRADED)
print("ball-missing gaps inside graded frames: %d | median %.2fs | p90 %.2fs | max %.1fs"
      % (len(gl), np.median(gl) / FPS, np.percentile(gl, 90) / FPS, gl.max() / FPS))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
minutes = np.arange(N) // (FPS * 60)
vis = pd.Series(ball_obs[GRADED]).groupby(minutes[GRADED]).mean()
ax[0].bar(vis.index, vis.values, width=0.9); ax[0].set_title("ball visible, share per match minute"); ax[0].set_ylim(0, 1)
ax[1].hist(gl / FPS, bins=np.logspace(-2, 2, 40)); ax[1].set_xscale("log"); ax[1].set_title("ball-missing gap length (s)")
ax[2].hist(n_home[GRADED], bins=np.arange(-.5, 30), alpha=.6, label="home"); ax[2].hist(n_away[GRADED], bins=np.arange(-.5, 30), alpha=.6, label="away")
ax[2].legend(); ax[2].set_title("tracked players per frame"); plt.tight_layout(); plt.show()

ball visible: all frames 56.9% | graded frames 59.3%
tracked players per frame (graded): home 10.2, away 10.3
ball observed OUTSIDE the pitch (+0.5 m) on 2.10% of graded frames
ball-missing gaps inside graded frames: 150 | median 10.72s | p90 33.81s | max 221.5s


<Figure size 1500x340 with 3 Axes>

In [7]:
import baseline as bl     # starter/baseline.py: reuse the organisers' smoothing function unchanged

def nearest_team_codes(P, B, slot_team, chunk=20000):
    """Per frame: team (0 home / 1 away) of the tracked player nearest to the raw ball; -1 if no answer."""
    out = np.full(len(B), -1, np.int8)
    for a in range(0, len(B), chunk):
        sl = slice(a, a + chunk)
        d = np.hypot(P[sl, :, 0] - B[sl, None, 0], P[sl, :, 1] - B[sl, None, 1])
        d = np.where(np.isnan(d), np.inf, d)
        j = d.argmin(1)
        ok = np.isfinite(d[np.arange(len(j)), j])
        out[sl][ok] = slot_team[j[ok]]
    return out

def evaluate_labels(labels, bootstrap_n=0):
    return pm.evaluate(GT_CODES, pm.label_codes(labels), bootstrap_n=bootstrap_n)

def headline(res, title=""):
    print("%-28s spellF1 %.4f | macroF1 %.4f | alive/dead F1 %.3f | team acc (GT alive) %.3f | possessions pred %d vs GT %d | mean dur %.1fs vs %.1fs"
          % (title, res["spell_f1_mean"], res["macro_f1"], res["alive_dead_f1"], res["team_acc_gt_alive"],
             res["n_pred_spells"], res["n_gt_spells"], res["mean_spell_duration_s"], res["gt_mean_spell_duration_s"]))

near = nearest_team_codes(P, B, slot_team)
for w in (0, 25, 50, 125):
    headline(evaluate_labels(bl.smooth(near, w)), "baseline window +/-%d" % w)
print("\nREAL-data reference (public window): smooth50 spellF1 0.3256 | smooth125 spellF1 0.4242 | raw ~0.05 on the full match")

baseline window +/-0         spellF1 0.0378 | macroF1 0.5972 | alive/dead F1 0.892 | team acc (GT alive) 0.728 | possessions pred 131 vs GT 17 | mean dur 2.0s vs 14.2s
baseline window +/-25        spellF1 0.2868 | macroF1 0.6200 | alive/dead F1 0.897 | team acc (GT alive) 0.791 | possessions pred 36 vs GT 17 | mean dur 7.6s vs 14.2s
baseline window +/-50        spellF1 0.3256 | macroF1 0.6207 | alive/dead F1 0.899 | team acc (GT alive) 0.835 | possessions pred 26 vs GT 17 | mean dur 10.8s vs 14.2s
baseline window +/-125       spellF1 0.4242 | macroF1 0.5812 | alive/dead F1 0.899 | team acc (GT alive) 0.885 | possessions pred 16 vs GT 17 | mean dur 18.5s vs 14.2s

REAL-data reference (public window): smooth50 spellF1 0.3256 | smooth125 spellF1 0.4242 | raw ~0.05 on the full match


## Ball cleaning and per-frame evidence

In [8]:
def nan_box_smooth(x, w):
    if w <= 1:
        return x
    valid = ~np.isnan(x); v = np.where(valid, x, 0.0); k = np.ones(w)
    num, den = np.convolve(v, k, "same"), np.convolve(valid.astype(float), k, "same")
    out = np.where(den > 0, num / np.maximum(den, 1e-9), np.nan)
    out[~valid] = np.nan
    return out

def clean_ball(B, fps, cfg):
    """Returns ball (N,2) metres with NaN where unknown, ball_state (N,) in {0,1,2}, ball_speed (N,) m/s."""
    n = len(B); Bc = B.copy()
    # 1) isolated spikes (two passes)
    for _ in range(2):
        idx = np.flatnonzero(~np.isnan(Bc[:, 0]))
        if len(idx) < 3:
            break
        step = np.hypot(*np.diff(Bc[idx], axis=0).T) / (np.diff(idx) / fps)       # speed between consecutive observations
        spike = np.zeros(len(idx), bool)
        spike[1:-1] = (step[:-1] > cfg["max_ball_speed"]) & (step[1:] > cfg["max_ball_speed"])
        Bc[idx[spike]] = np.nan
    obs = ~np.isnan(Bc[:, 0])
    state = obs.astype(np.int8)
    # 2) interpolate short gaps that are bracketed by observations
    idx = np.flatnonzero(obs)
    if len(idx) >= 2:
        gs, gl = run_lengths(~obs)
        max_gap = int(round(cfg["max_gap_s"] * fps))
        for s, l in zip(gs, gl):
            if l <= max_gap and s > idx[0] and s + l - 1 < idx[-1]:
                sl = slice(s, s + l)
                for c in (0, 1):
                    Bc[sl, c] = np.interp(np.arange(s, s + l), idx, Bc[idx, c])
                state[sl] = 2
    # 3) smooth + velocity
    Bs = np.stack([nan_box_smooth(Bc[:, c], cfg["smooth_frames"]) for c in (0, 1)], axis=1)
    v = np.full(n, np.nan)
    ok = ~np.isnan(Bs[:, 0])
    dx, dy = np.gradient(np.where(ok, Bs[:, 0], np.nan)), np.gradient(np.where(ok, Bs[:, 1], np.nan))
    v[ok] = np.hypot(dx, dy)[ok] * fps
    return Bs, state, v

ball, ball_state, ball_speed = clean_ball(B, FPS, CFG)
print("graded frames: raw %.1f%% | after cleaning %.1f%% (observed %.1f%% + interpolated %.1f%%)"
      % (100 * ball_obs[GRADED].mean(), 100 * (ball_state[GRADED] > 0).mean(),
         100 * (ball_state[GRADED] == 1).mean(), 100 * (ball_state[GRADED] == 2).mean()))
print("ball speed (m/s) graded: median %.1f | p95 %.1f | p99 %.1f" % tuple(np.nanpercentile(ball_speed[GRADED], [50, 95, 99])))

graded frames: raw 59.3% | after cleaning 59.3% (observed 59.2% + interpolated 0.1%)
ball speed (m/s) graded: median 5.1 | p95 17.3 | p99 24.3


In [9]:
def team_of_tag(text, cfg):
    t = (text or "").lower()
    if any(k in t for k in cfg["st_home_keys"]): return +1
    if any(k in t for k in cfg["st_away_keys"]): return -1
    return 0                                               # "N/A", empty, unknown

def smart_tagging_arrays(ST, n, cfg):
    st_team = np.zeros(n); st_sp = np.zeros(n)
    for r in ST.itertuples():
        a, b = max(r.start, 0), min(r.end, n)
        if b <= a:
            continue
        if r.code == "SET PIECES":
            st_sp[a:b] = 1.0
        if r.code in cfg["st_skip_codes"]:
            continue
        sign = team_of_tag(r.team, cfg)
        if r.code.startswith(tuple(cfg["st_flip_prefixes"])):
            sign = -sign
        st_team[a:b] += sign
    return np.clip(st_team, -1, 1), st_sp

def compute_features(cfg, ball, ball_state, ball_speed, chunk=20000):
    n = len(ball)
    ch, ca = np.zeros(n), np.zeros(n)
    home, away = slot_team == 0, slot_team == 1
    for a in range(0, n, chunk):                                           # chunked: N x slots matrices are big
        sl = slice(a, a + chunk)
        d = np.hypot(P[sl, :, 0] - ball[sl, None, 0], P[sl, :, 1] - ball[sl, None, 1])
        k = np.nan_to_num(np.exp(-(d ** 2) / (2 * cfg["sigma_m"] ** 2)), nan=0.0)   # NaN distance (no ball / no player) -> 0
        ch[sl], ca[sl] = k[:, home].sum(1), k[:, away].sum(1)
    eps = 1e-3
    ev = np.clip(np.log((ch + eps) / (ca + eps)), -cfg["evidence_clip"], cfg["evidence_clip"])
    ev[(ch + ca) < cfg["min_control"]] = 0.0                               # loose ball: no team evidence
    ev[ball_state == 0] = 0.0                                              # no ball: no evidence
    ball_w = np.where(ball_state == 1, 1.0, np.where(ball_state == 2, cfg["interp_weight"], 0.0))
    std_ = globals().get("BALL_STD")                          # set by Step 4b when the Kalman smoother is used
    if std_ is not None:                                      # filled frames count in proportion to their certainty
        conf = np.clip(1.0 - np.nan_to_num(std_, nan=cfg["kf_std_cap"]) / cfg["kf_std_cap"], 0.0, 1.0)
        ball_w = np.where(ball_state == 2, cfg["interp_weight"] * conf, ball_w)

    m = cfg["pitch_margin_m"]
    present = ball_state > 0
    out = (present & ((ball[:, 0] < -m) | (ball[:, 0] > L_PITCH + m) | (ball[:, 1] < -m) | (ball[:, 1] > W_PITCH + m))).astype(float)
    win = max(int(round(cfg["static_window_s"] * FPS)), 1)
    roll_max = pd.Series(ball_speed).rolling(win, center=True, min_periods=win // 2).max().values
    static = (present & (roll_max < cfg["static_speed"])).astype(float)
    gs, gl = run_lengths(ball_state == 0)
    missing_long = np.zeros(n)
    for s, l in zip(gs, gl):
        if l >= cfg["missing_min_s"] * FPS:
            missing_long[s:s + l] = 1.0
    st_team, st_sp = smart_tagging_arrays(ST, n, cfg)
    return dict(ev=ev, ball_w=ball_w, out=out, static=static, missing_long=missing_long,
                st_team=st_team, st_sp=st_sp, ch=ch, ca=ca)

t0 = time.time()
FEATS = compute_features(CFG, ball, ball_state, ball_speed)
print("features computed in %.0fs" % (time.time() - t0))
print(pd.DataFrame({k: pd.Series(v[GRADED]).describe()[["mean", "std", "min", "max"]] for k, v in FEATS.items()}).T.round(3))

features computed in 0s
               mean    std  min    max
ev            0.179  1.846 -4.0  4.000
ball_w        0.593  0.491  0.0  1.000
out           0.021  0.143  0.0  1.000
static        0.022  0.145  0.0  1.000
missing_long  0.406  0.491  0.0  1.000
st_team       0.186  0.839 -1.0  1.000
st_sp         0.276  0.447  0.0  1.000
ch            0.341  0.468  0.0  3.667
ca            0.299  0.466  0.0  5.463


## Learned home/away model (weak labels over the whole match, cross-fitted)

In [10]:
ML_TASKS = ("team",)                    # the dead-ball model is fitted to the real labels below, not to the tags
ML_STRIDE = 5                            # train on every 5th labelled frame (neighbours are near-duplicates)
ML_BLOCK_S = 90                          # seconds per time block for group folds
ML_FOLDS = 5
ML_SEED = 0
ML_LOGIT_CLIP = 4.0

CFG.update(use_ml=False,                 # switch on after you have looked at the diagnostics in Step 6b
           w_ml_team=0.10,               # weight of the learned team log-odds in the H/A emission
           w_ml_dead=0.30,               # weight of the learned dead log-odds in the D emission
           ml_dead_offset=0.0)           # subtract from the learned dead log-odds (shifts its prior)

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss

# ---- reproducibility: the two learned models are pinned, so the same file is written on any machine.
# team (weak-label home/away model): Linear Discriminant Analysis, default settings | dead (dead vs alive): Extra Trees, parameters below.
# Both were selected once by a cross-validated comparison; the notebook still prints the pin line of whatever it uses (pin_hint).
# None = re-run the scikit-learn selection (deterministic for fixed library versions, but not guaranteed to pick the same model).
PIN_ESTIMATORS = {"team": "LinearDiscriminantAnalysis",
                  "dead": ("ExtraTreesClassifier", {"min_samples_leaf": 5, "n_estimators": 200, "n_jobs": 1, "random_state": 0})}
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
KNOWN_EST = {"LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(), "LogisticRegression": LogisticRegression(max_iter=1000),
             "ExtraTreesClassifier": ExtraTreesClassifier(n_estimators=100, random_state=0, n_jobs=1),
             "RandomForestClassifier": RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=1),
             "HistGradientBoostingClassifier": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=150, l2_regularization=1.0, random_state=0)}

def make_pinned(pin):
    """pin = "ClassName" (default settings) or ("ClassName", {non-default params}) exactly as printed by pin_hint()."""
    if isinstance(pin, str):
        return clone(KNOWN_EST[pin])
    name, params = pin
    return type(KNOWN_EST[name])(**params)

def pin_hint(task, est):
    """The exact line to paste into PIN_ESTIMATORS so that the same model is rebuilt on any machine (no model search needed)."""
    name = type(est).__name__
    try:
        default = type(est)().get_params()
    except Exception:
        default = {}
    diff = {k: v for k, v in est.get_params().items() if k not in default or default[k] != v}
    return '   to pin it for the final run:  PIN_ESTIMATORS["%s"] = %r' % (task, (name, diff) if diff else name)


In [11]:
def roll(x, win, how="mean", center=True):
    r = pd.Series(x).rolling(win, center=center, min_periods=1)
    return getattr(r, how)().values

def build_ml_table(ball, ball_state, ball_speed, F, chunk=20000):
    """(N, F) feature matrix for the learned emission model. NO Smart Tagging inputs (they are the label source)."""
    n = len(ball); home, away = slot_team == 0, slot_team == 1
    dh, da = np.full(n, 40.0), np.full(n, 40.0)
    cnt = {k: np.zeros(n) for k in ("n3h", "n3a", "n6h", "n6a", "n10h", "n10a", "nh", "na")}
    pspd = np.zeros(n)
    for a in range(0, n, chunk):
        sl = slice(a, a + chunk)
        d = np.hypot(P[sl, :, 0] - ball[sl, None, 0], P[sl, :, 1] - ball[sl, None, 1])
        d = np.where(np.isnan(d), np.inf, d)
        for tag, m in (("h", home), ("a", away)):
            dm = d[:, m]
            best = dm.min(1) if dm.shape[1] else np.full(len(d), np.inf)
            (dh if tag == "h" else da)[sl] = np.where(np.isfinite(best), np.minimum(best, 40.0), 40.0)
            for r in (3, 6, 10):
                cnt["n%d%s" % (r, tag)][sl] = (dm < r).sum(1)
            cnt["n" + tag][sl] = (~np.isnan(P[sl][:, m, 0])).sum(1)
        sp = PSPD[sl]; ok = ~np.isnan(sp)
        pspd[sl] = np.where(ok.sum(1) > 0, np.where(ok, sp, 0).sum(1) / np.maximum(ok.sum(1), 1), 0)
    present = ball_state > 0
    ev_s = F["ev"] * F["ball_w"]
    dd = np.clip(dh - da, -40, 40)
    bs = np.nan_to_num(ball_speed)
    bx, by = np.nan_to_num(ball[:, 0]), np.nan_to_num(ball[:, 1])
    x_c = np.where(present, np.abs(bx - L_PITCH / 2) / (L_PITCH / 2), 0)
    y_c = np.where(present, np.abs(by - W_PITCH / 2) / (W_PITCH / 2), 0)
    edge = np.where(present, np.minimum.reduce([bx, L_PITCH - bx, by, W_PITCH - by]), 0)
    cols = dict(dh=dh, da=da, dd=dd, ch=F["ch"], ca=F["ca"], nh=cnt["nh"], na=cnt["na"],
                n3h=cnt["n3h"], n3a=cnt["n3a"], n6h=cnt["n6h"], n6a=cnt["n6a"], n10h=cnt["n10h"], n10a=cnt["n10a"],
                ev=ev_s, ev_m25=roll(ev_s, 51), ev_m75=roll(ev_s, 151),
                ev_past75=roll(ev_s, 76, center=False), ev_fut75=roll(ev_s[::-1], 76, center=False)[::-1],
                dd_m25=roll(dd, 51), dd_m75=roll(dd, 151),
                bspd=bs, bacc=np.abs(np.gradient(bs)), bspd_max25=roll(bs, 51, "max"), bspd_mean75=roll(bs, 151),
                static_m50=roll(F["static"], 101), missing_m75=roll(1.0 - present, 151),
                b_edge=edge, b_cx=x_c, b_cy=y_c, pspd=pspd,
                ball_missing=(ball_state == 0).astype(float), ball_interp=(ball_state == 2).astype(float), out=F["out"])
    names = list(cols)
    X = np.stack([np.nan_to_num(np.asarray(cols[k], float)) for k in names], axis=1)
    return X, names

SWAP_PAIRS = [("dh", "da"), ("ch", "ca"), ("nh", "na"), ("n3h", "n3a"), ("n6h", "n6a"), ("n10h", "n10a")]
NEGATE = ["dd", "ev", "ev_m25", "ev_m75", "ev_past75", "ev_fut75", "dd_m25", "dd_m75"]

ML_X, ML_COLS = build_ml_table(ball, ball_state, ball_speed, FEATS)
ML_RAW = ML_X.astype(np.float32)          # unscaled copy, used by the model lab (Step 5c)
ci = {c: i for i, c in enumerate(ML_COLS)}

# Scale without breaking the home/away symmetry: pooled std for swapped pairs, std-only for signed columns.
scale = np.ones(len(ML_COLS)); shift = np.zeros(len(ML_COLS)); G = GRADED
for a_, b_ in SWAP_PAIRS:
    s = np.concatenate([ML_X[G, ci[a_]], ML_X[G, ci[b_]]]).std() + 1e-9
    scale[ci[a_]] = scale[ci[b_]] = s
for c in NEGATE:
    scale[ci[c]] = ML_X[G, ci[c]].std() + 1e-9
for c in ML_COLS:
    if c not in NEGATE and not any(c in p for p in SWAP_PAIRS):
        shift[ci[c]] = ML_X[G, ci[c]].mean(); scale[ci[c]] = ML_X[G, ci[c]].std() + 1e-9
ML_X = ((ML_X - shift) / scale).astype(np.float32)

def swap_teams(X):
    Y_ = X.copy()
    for a_, b_ in SWAP_PAIRS:
        Y_[:, ci[a_]], Y_[:, ci[b_]] = X[:, ci[b_]], X[:, ci[a_]]
    for c in NEGATE:
        Y_[:, ci[c]] = -X[:, ci[c]]
    return Y_

print("ML feature matrix:", ML_X.shape, "| columns:", len(ML_COLS))
print("symmetry check (swap twice == identity):", bool(np.allclose(swap_teams(swap_teams(ML_X[:1000])), ML_X[:1000])))

ML feature matrix: (199319, 33) | columns: 33
symmetry check (swap twice == identity): True


In [12]:
from scipy.spatial import ConvexHull, cKDTree

def spatial_columns():
    n = N
    hull_h, hull_a = np.zeros(n), np.zeros(n)
    cd_h, cd_a = np.full(n, 40.0), np.full(n, 40.0)
    nn_gap, nn_mixed = np.full(n, 20.0), np.zeros(n)
    home, away = slot_team == 0, slot_team == 1
    for f in np.flatnonzero(GRADED):
        ok = ~np.isnan(P[f, :, 0])
        for m, hull, cd in ((home, hull_h, cd_h), (away, hull_a, cd_a)):
            pts = P[f][ok & m]
            if len(pts) >= 3:
                try:
                    hull[f] = ConvexHull(pts).volume                      # 2-D: .volume is the area
                except Exception:                                          # collinear points
                    pass
            if len(pts) and ball_state[f] > 0:
                cd[f] = min(float(np.hypot(*(pts.mean(0) - ball[f]))), 40.0)
        if ball_state[f] > 0 and ok.sum() >= 2:
            tree = cKDTree(P[f][ok])
            d, j = tree.query(ball[f], k=2)
            nn_gap[f] = min(float(d[1] - d[0]), 20.0)
            nn_mixed[f] = float(slot_team[np.flatnonzero(ok)[j[0]]] != slot_team[np.flatnonzero(ok)[j[1]]])
    return dict(hull_h=hull_h, hull_a=hull_a, cdist_h=cd_h, cdist_a=cd_a, nn_gap=nn_gap, nn_mixed=nn_mixed)

def add_ml_columns(new_cols, pairs=()):
    """Append raw columns to ML_RAW and scaled columns to ML_X; register home/away pairs for the swap augmentation."""
    global ML_X, ML_RAW, ML_COLS, ci, SWAP_PAIRS
    names = list(new_cols)
    raw = np.stack([np.nan_to_num(np.asarray(new_cols[k], float)) for k in names], axis=1)
    scaled = raw.copy()
    paired = {c for p in pairs for c in p}
    for a_, b_ in pairs:
        ia, ib = names.index(a_), names.index(b_)
        s = np.concatenate([raw[GRADED, ia], raw[GRADED, ib]]).std() + 1e-9      # pooled std keeps the swap symmetry
        scaled[:, ia] /= s; scaled[:, ib] /= s
    for k in names:
        if k not in paired:
            j = names.index(k); scaled[:, j] = (raw[:, j] - raw[GRADED, j].mean()) / (raw[GRADED, j].std() + 1e-9)
    ML_RAW = np.hstack([ML_RAW, raw.astype(np.float32)]); ML_X = np.hstack([ML_X, scaled.astype(np.float32)])
    ML_COLS = ML_COLS + names; ci = {c: i for i, c in enumerate(ML_COLS)}
    SWAP_PAIRS = SWAP_PAIRS + list(pairs)

t0 = time.time()
SPATIAL_COLS = spatial_columns()
add_ml_columns(SPATIAL_COLS, pairs=[("hull_h", "hull_a"), ("cdist_h", "cdist_a")])
print("added %d scipy.spatial columns in %.0fs -> ML feature matrix %s" % (len(SPATIAL_COLS), time.time() - t0, ML_X.shape))
print("symmetry check:", bool(np.allclose(swap_teams(swap_teams(ML_X[:1000])), ML_X[:1000])))

added 6 scipy.spatial columns in 21s -> ML feature matrix (199319, 39)
symmetry check: True


In [13]:
# Weak labels from Smart Tagging (the ONLY use of the tags in this step) and time blocks.
st_team_f, st_sp_f = FEATS["st_team"], FEATS["st_sp"]
Y = {
    "team": np.where(GRADED & (st_sp_f == 0) & (st_team_f > 0.5), 1, np.where(GRADED & (st_sp_f == 0) & (st_team_f < -0.5), 0, -1)),
    "dead": np.where(GRADED & (st_sp_f == 1), 1, np.where(GRADED & (st_sp_f == 0) & (st_team_f != 0), 0, -1)),
}
FRAMES = np.arange(N)
BLOCK = FRAMES // int(ML_BLOCK_S * FPS)
FOLD = (BLOCK % ML_FOLDS)
TRAIN_ROWS = {t: np.flatnonzero((Y[t] >= 0) & (FRAMES % ML_STRIDE == 0)) for t in ML_TASKS}
for t in ML_TASKS:
    r = TRAIN_ROWS[t]
    print("%-5s weak-labelled frames: %6d (training rows at stride %d: %5d) | positive share %s | blocks %d"
          % (t, int((Y[t] >= 0).sum()), ML_STRIDE, len(r), ("%.2f" % Y[t][r].mean()) if len(r) else "n/a", len(np.unique(BLOCK[r]))))

def training_frame(task):
    """Time-ordered training table (home/away-swapped copies added for the team task)."""
    rows = TRAIN_ROWS[task]
    X, y, fr = ML_X[rows], Y[task][rows], rows
    if task == "team":
        X = np.vstack([X, swap_teams(X)]); y = np.r_[y, 1 - y]; fr = np.r_[fr, fr]
        order = np.argsort(fr, kind="stable"); X, y, fr = X[order], y[order], fr[order]
    df = pd.DataFrame(X, columns=ML_COLS); df["target"] = y.astype(int); df["block"] = BLOCK[fr]
    return df

SK_CANDIDATES = {
    "logreg": LogisticRegression(C=0.5, max_iter=500),
    "hist_gbm": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200,
                                               l2_regularization=1.0, random_state=ML_SEED),
    "extra_trees": ExtraTreesClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=1, random_state=ML_SEED),
}

def select_sklearn(df):
    """Compare candidates with time-block GroupKFold; pick the lowest out-of-fold log-loss."""
    Xd, yd, gd = df[ML_COLS].values, df["target"].values, df["block"].values
    rows = []
    for name, est in SK_CANDIDATES.items():
        oof = np.zeros(len(df))
        for tr, te in GroupKFold(ML_FOLDS).split(Xd, yd, gd):
            oof[te] = clone(est).fit(Xd[tr], yd[tr]).predict_proba(Xd[te])[:, 1]
        rows.append(dict(model=name, auc=roc_auc_score(yd, oof), logloss=log_loss(yd, np.clip(oof, 1e-4, 1 - 1e-4))))
    board = pd.DataFrame(rows).sort_values("logloss").reset_index(drop=True)
    return clone(SK_CANDIDATES[board.loc[0, "model"]]), board

def enough_data(df):
    return len(df) >= 200 and df["target"].nunique() == 2 and df["block"].nunique() >= ML_FOLDS

SELECTED, BOARDS = {}, {}
for t in ML_TASKS:
    df = training_frame(t)
    if not enough_data(df):
        print("[%s] not enough weak labels (%d rows, %d blocks, %d classes) -> task skipped, emissions stay at 0"
              % (t, len(df), df["block"].nunique(), df["target"].nunique()))
        SELECTED[t] = None
        continue
    if PIN_ESTIMATORS.get(t):                          # pinned: identical on every machine
        SELECTED[t], BOARDS[t] = make_pinned(PIN_ESTIMATORS[t]), pd.DataFrame({"model": [str(PIN_ESTIMATORS[t]) + " (pinned)"]})
    else:
        SELECTED[t], BOARDS[t] = select_sklearn(df)
    print("\n== task: %s | selected: %s ==" % (t, type(SELECTED[t]).__name__))
    print(pin_hint(t, SELECTED[t]))
    print(BOARDS[t].round(4).to_string(index=False))

team  weak-labelled frames:  84984 (training rows at stride 5: 16988) | positive share 0.62 | blocks 69

== task: team | selected: LinearDiscriminantAnalysis ==
   to pin it for the final run:  PIN_ESTIMATORS["team"] = 'LinearDiscriminantAnalysis'
                              model
LinearDiscriminantAnalysis (pinned)


In [14]:
def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))

def crossfit(est, task, X):
    """Predict every graded frame with a model that never saw that frame's time block. X = feature matrix."""
    p = np.full(N, np.nan); aug = task == "team"
    for f in range(ML_FOLDS):
        tr = TRAIN_ROWS[task][FOLD[TRAIN_ROWS[task]] != f]
        te = np.flatnonzero(GRADED & (FOLD == f))
        if not len(te) or not len(tr):
            continue
        Xtr, ytr = X[tr], Y[task][tr]
        if aug:
            Xtr, ytr = np.vstack([Xtr, swap_teams(Xtr)]), np.r_[ytr, 1 - ytr]
        m = clone(est).fit(Xtr, ytr)
        pr = m.predict_proba(X[te])[:, 1]
        if aug:                                            # symmetric prediction
            pr = 0.5 * (pr + 1 - m.predict_proba(swap_teams(X[te]))[:, 1])
        p[te] = pr
    return p

def build_emissions(selected, matrices=None, verbose=True):
    """Cross-fit + calibrate every selected model. Returns {'ml_team': log-odds, 'ml_dead': log-odds} (zeros if skipped).
    matrices: optional {task: feature matrix}; default is the standardised ML_X (use ML_RAW for pipelines that scale themselves)."""
    matrices = matrices or {}
    out = {"ml_team": np.zeros(N), "ml_dead": np.zeros(N)}
    for t in ML_TASKS:
        if selected.get(t) is None:
            continue
        raw = crossfit(selected[t], t, matrices.get(t, ML_X))
        lab = Y[t] >= 0
        z = logit(raw[lab]).reshape(-1, 1)
        platt = LogisticRegression(C=1e6, fit_intercept=(t != "team")).fit(z, Y[t][lab])      # team: no intercept (symmetry)
        cal = np.full(N, np.nan); ok = GRADED & ~np.isnan(raw)
        cal[ok] = platt.predict_proba(logit(raw[ok]).reshape(-1, 1))[:, 1]
        out["ml_" + t] = np.nan_to_num(np.clip(logit(cal), -ML_LOGIT_CLIP, ML_LOGIT_CLIP))
        if verbose:
            print("%-5s cross-fitted AUC vs weak labels %.3f | Platt slope %.2f (below 1 = the raw model was overconfident)"
                  % (t, roc_auc_score(Y[t][lab], raw[lab]), platt.coef_[0, 0]))
    return out

EMIS = {}                                                   # name -> emission arrays; compared in Step 6c
EMIS["5b"] = build_emissions(SELECTED)
FEATS.update(EMIS["5b"])
print("\nadded FEATS['ml_team'], FEATS['ml_dead'] (log-odds, clipped to +/-%.0f); stored as EMIS['5b']" % ML_LOGIT_CLIP)

team  cross-fitted AUC vs weak labels 0.887 | Platt slope 0.85 (below 1 = the raw model was overconfident)

added FEATS['ml_team'], FEATS['ml_dead'] (log-odds, clipped to +/-4); stored as EMIS['5b']


## Decoder building blocks

In [15]:
LETTER = np.array(["H", "A", "D"])

def build_scores(F, cfg):
    ctrl = cfg["w_ctrl"] * F["ev"] * F["ball_w"] + cfg["w_st_team"] * F["st_team"]
    dead = (-cfg["dead_bias"] + cfg["w_out"] * F["out"] + cfg["w_static"] * F["static"]
            + cfg["w_setpiece"] * F["st_sp"] + cfg["w_missing"] * F["missing_long"])
    for _name, _w in (cfg.get("dead_cues") or {}).items():   # optional extra dead-ball cues (standardised feature columns)
        dead = dead + _w * ML_X[:, ci[_name]]
    if cfg.get("use_ml") and "ml_team" in F:               # learned emissions from Step 5b (optional)
        ctrl = ctrl + cfg["w_ml_team"] * F["ml_team"]
        dead = dead + cfg["w_ml_dead"] * (F["ml_dead"] - cfg["ml_dead_offset"])
    return np.stack([ctrl, -ctrl, dead], axis=1)

def viterbi(scores, switch_cost, dead_cost):
    """Best state path for (T,3) per-frame scores (higher = better) with H/A/D transition penalties."""
    T = len(scores)
    trans = np.array([[0, -switch_cost, -dead_cost],
                      [-switch_cost, 0, -dead_cost],
                      [-dead_cost, -dead_cost, 0]], dtype=float)
    delta = np.empty((T, 3)); psi = np.empty((T, 3), np.int8)
    delta[0] = scores[0]
    for t in range(1, T):
        cand = delta[t - 1][:, None] + trans            # (previous state, current state)
        psi[t] = cand.argmax(0)
        delta[t] = cand.max(0) + scores[t]
    path = np.empty(T, np.int8); path[-1] = delta[-1].argmax()
    for t in range(T - 1, 0, -1):
        path[t - 1] = psi[t, path[t]]
    return path

def enforce_min_duration(path, min_alive, min_dead):
    """Repeatedly absorb the shortest too-short run into the longer neighbour (or the common one)."""
    path = path.copy()
    for _ in range(100000):
        change = np.flatnonzero(np.diff(path) != 0) + 1
        starts, ends = np.r_[0, change], np.r_[change, len(path)]
        lens, states = ends - starts, path[starts]
        if len(states) == 1:
            break
        bad = np.flatnonzero(lens < np.where(states == 2, min_dead, min_alive))
        if not len(bad):
            break
        k = bad[np.argmin(lens[bad])]
        if k == 0:
            new = states[1]
        elif k == len(states) - 1:
            new = states[k - 1]
        elif states[k - 1] == states[k + 1]:
            new = states[k - 1]
        else:
            new = states[k - 1] if lens[k - 1] >= lens[k + 1] else states[k + 1]
        path[starts[k]:ends[k]] = new
    return path

def decode_match(F, cfg):
    """Full-length label array ('H','A','D', and 'X' on ungraded frames) + per-half raw/final paths."""
    scores = build_scores(F, cfg)
    labels = np.full(N, "X", dtype="<U1"); paths = {}
    for half, a, b in HALVES:
        sc = scores[a:b + 1].copy()
        kick = MATCH["pre_kickoff_dead"].get(half)
        if kick:                                           # graded but before kickoff: all dead ball
            sc[:kick - a, :2] = -1e3
        if cfg.get("decoder") == "hsmm":                    # explicit duration model (Step 6d)
            kk = (kick - a) if kick else 0                  # the forced-dead stretch before kickoff is not decoded
            raw = final = np.r_[np.full(kk, 2, np.int8), hsmm_decode(sc[kk:], cfg)]
        else:
            raw = viterbi(sc, cfg["switch_cost"], cfg["dead_cost"])
            final = enforce_min_duration(raw, int(round(cfg["min_possession_s"] * FPS)), int(round(cfg["min_dead_s"] * FPS)))
        labels[a:b + 1] = LETTER[final]
        paths[half] = (raw, final)
    return labels, paths

t0 = time.time()
labels, paths = decode_match(FEATS, CFG)
print("decoded in %.1fs | label counts on graded frames:" % (time.time() - t0),
      {l: int(((labels == l) & GRADED).sum()) for l in "HAD"})

decoded in 0.8s | label counts on graded frames: {'H': 74525, 'A': 49371, 'D': 34961}


## Final model — calibrated home/away and dead/alive, three emission designs, honest selection

In [16]:
from itertools import product
from sklearn.model_selection import LeaveOneGroupOut

FINAL_DEAD_TARGET = 0.38      # assumed whole-match dead share (30-45% is typical football). The DECODED output is fitted to it.
FINAL_C = 0.1                 # logistic regression regularisation
FINAL_BLOCK_S = 60            # leave-one-block-out block length (s)
FINAL_DEAD_MODES = ["relative", "ball_relative", "learned"]      # "learned" = a nonlinear estimator on the ball_relative cues (defined in the next cell)
FINAL_TOPK = 30               # per candidate: the settings with the best frame F1 are re-scored on spellF1 and the best of them is kept
FINAL_WEIGHT_SPELL = 0.5      # candidate score = W * spellF1 + (1 - W) * nested frame metric   (spellF1 is the ranking metric, but it is noisy on 17 possessions)
FINAL_CONSISTENCY = 1.0       # reject a candidate if |whole-match dead shift - window dead offset| exceeds this (mis-specified for the match)
FINAL_FRAME_METRIC = "Accuracy"   # frame metric inside the candidate score: "Accuracy" | "F1"
FINAL_TEAM_MODEL = "auto"     # "auto" or the letter of a team variant (A-F) from the comparison table
FINAL_EMISSIONS = ["hierarchical", "balanced", "mixed"]
FINAL_FALLBACK = ("balanced", "relative")                         # structure already validated on the real data
FINAL_SANITY = dict(dead_stretches_per_min=(0.55, 2.1), dead_mean_s=(8, 45), poss_per_min=(1.0, 4.5), poss_mean_s=(8, 40))   # per-minute rates: ~60-220 dead stretches in a full match
FINAL_GRID = (dict(tau_t=[0.2, 0.4], tau_d=[0.2, 0.4], switch=[4], dead_cost=[6, 20], offset=[0.0, 1.0]) if SYNTHETIC else      # dry-run: small grid
              dict(tau_t=[0.1, 0.2, 0.4], tau_d=[0.1, 0.2, 0.4], switch=[2, 4, 8], dead_cost=[6, 12, 20, 30], offset=[-1.0, -0.5, 0.0, 0.5, 1.0, 2.0]))

def _roll(x, w):
    return pd.Series(x).rolling(w, center=True, min_periods=1).mean().values
def _sig(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
def _mat(d, center):
    names = list(d); X_ = np.stack([np.nan_to_num(np.asarray(d[k], float)) for k in names], axis=1)
    mu = X_[GRADED].mean(0) if center else np.zeros(X_.shape[1]); sd = X_[GRADED].std(0); sd[sd < 1e-9] = 1.0
    return (X_ - mu) / sd, names

def frame_metrics(labels_):
    r = evaluate_labels(labels_)
    cm = np.array(r["confusion"], float)[:3, :3]; tp = np.diag(cm)
    rec = tp / np.maximum(cm.sum(1), 1); prec = tp / np.maximum(cm.sum(0), 1); f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return dict(Accuracy=tp.sum() / max(cm.sum(), 1), Precision=prec.mean(), Recall=rec.mean(), F1=f1.mean(), spellF1=r["spell_f1_mean"],
                macroF1=r["macro_f1"], possessions=r["n_pred_spells"], truth_possessions=r["n_gt_spells"])

# ---------------- team cues (all antisymmetric: positive = home)
dh_, da_ = ML_RAW[:, ci["dh"]].astype(float), ML_RAW[:, ci["da"]].astype(float)
have_ = (ML_RAW[:, ci["ball_missing"]] == 0) & (np.minimum(dh_, da_) < 15)
ns_ = np.where(have_, np.sign(da_ - dh_), 0.0)
hh_, ha_ = ML_RAW[:, ci["hull_h"]].astype(float), ML_RAW[:, ci["hull_a"]].astype(float)
TEAM3 = {"learned team model (weak labels)": EMIS["5b"]["ml_team"], "nearest-player vote +/-5s": _roll(ns_, 251),
         "control evidence +/-3s": ML_RAW[:, ci["ev_m75"]].astype(float), "team shape: hull difference +/-2s": _roll((hh_ - ha_) / (hh_ + ha_ + 1.0), 101),
         "smart-tagging team prior": FEATS["st_team"]}
XT3, TEAM3_NAMES = _mat(TEAM3, False)

# ---------------- dead cues: player behaviour relative to the local tempo (+ set-piece tag), optionally ball cues
ok_ = ~np.isnan(PSPD); cnt_ = ok_.sum(1)
pmean_ = np.where(cnt_ > 0, np.where(ok_, PSPD, 0).sum(1) / np.maximum(cnt_, 1), 0.0)
slow_ = np.where(cnt_ > 0, ((PSPD < 1.5) & ok_).sum(1) / np.maximum(cnt_, 1), 0.0)
present_ = (ball_state > 0).astype(float)
bspd_seen = np.nan_to_num(pd.Series(np.where(present_ > 0, np.nan_to_num(ball_speed), np.nan)).rolling(151, center=True, min_periods=1).mean().values, nan=5.0)
bx_, by_ = np.nan_to_num(ball[:, 0]), np.nan_to_num(ball[:, 1])
edge_ = np.where(present_ > 0, np.clip(np.minimum.reduce([bx_, L_PITCH - bx_, by_, W_PITCH - by_]), 0, 20), 10.0)
def _lrange(wf):
    sx, sy = pd.Series(np.where(present_ > 0, ball[:, 0], np.nan)), pd.Series(np.where(present_ > 0, ball[:, 1], np.nan)); mp = max(int(0.5 * wf), 2)
    rx = sx.rolling(wf, center=True, min_periods=mp).max() - sx.rolling(wf, center=True, min_periods=mp).min()
    ry = sy.rolling(wf, center=True, min_periods=mp).max() - sy.rolling(wf, center=True, min_periods=mp).min()
    return np.log1p(np.nan_to_num(np.maximum(rx, ry).values, nan=3.0))
def _local_base(x, q=0.7, half_s=300, step=25):
    idx = np.flatnonzero(GRADED); xs = x[idx][::step]
    b = pd.Series(xs).rolling(2 * half_s + 1, center=True, min_periods=30).quantile(q).values
    b = pd.Series(b).bfill().ffill().values
    out = np.zeros(len(x)); out[idx] = np.interp(np.arange(len(idx)), np.arange(0, len(idx), step)[:len(b)], b)
    return out
_p2, _s2, _h2 = _roll(pmean_, 101), _roll(slow_, 101), np.log1p(_roll(hh_ + ha_, 101))
DEAD = {"rel": {"player speed vs local tempo": _p2 / (_local_base(_p2) + 0.1), "slow players vs local tempo": _s2 - _local_base(_s2, q=0.3),
                "team spread vs local baseline": _h2 - _local_base(_h2)},
        "tag": {"set-piece tag": FEATS["st_sp"]},
        "ball": {"ball outside pitch": FEATS["out"], "ball speed (seen frames) +/-3s": bspd_seen, "ball distance to the lines": edge_, "ball range +/-2.5s (log)": _lrange(125)}}
MODES = {"relative": ["rel", "tag"], "ball_relative": ["rel", "tag", "ball"]}
XD, DEAD_NAMES = {}, {}
for mode in [m for m in FINAL_DEAD_MODES if m != "learned"]:
    XD[mode], DEAD_NAMES[mode] = _mat({k: v for g in MODES[mode] for k, v in DEAD[g].items()}, True)

# ---------------- the labelled window and leave-one-minute-out logits
runs_v3 = pm.valid_runs(GT_CODES)
v3_lab = np.concatenate([np.arange(a_, b_) for a_, b_ in runs_v3]); n_lab = len(v3_lab)
v3_h, v3_a, v3_d = (GT_CODES == pm.H), (GT_CODES == pm.A), (GT_CODES == pm.D)
blocks_lab = np.arange(n_lab) // int(FINAL_BLOCK_S * FPS); nb_ = int(blocks_lab.max()) + 1
tc_lab = GT_CODES[v3_lab].astype(int)

def fit_team(frames):
    f = frames[(v3_h | v3_a)[frames]]
    return LogisticRegression(C=FINAL_C, fit_intercept=False, max_iter=3000).fit(XT3[f], v3_h[f].astype(int))
def fit_dead(frames, mode):
    return LogisticRegression(C=FINAL_C, fit_intercept=True, max_iter=3000).fit(XD[mode][frames], v3_d[frames].astype(int))

LIN_MODES = [m for m in FINAL_DEAD_MODES if m != "learned"]
zt_oof, zd_oof = np.zeros(N), {m: np.zeros(N) for m in LIN_MODES}
for tr_b, te_b in LeaveOneGroupOut().split(v3_lab, groups=blocks_lab):
    tr_f, te_f = v3_lab[tr_b], v3_lab[te_b]
    zt_oof[te_f] = XT3[te_f] @ fit_team(tr_f).coef_.ravel()
    for mode in LIN_MODES:
        zd_oof[mode][te_f] = fit_dead(tr_f, mode).decision_function(XD[mode][te_f])
mt_final = fit_team(v3_lab); zt_final = XT3 @ mt_final.coef_.ravel()
md_final = {m: fit_dead(v3_lab, m) for m in LIN_MODES}; zd_final = {m: md_final[m].decision_function(XD[m]) for m in LIN_MODES}
tm = v3_lab[(v3_h | v3_a)[v3_lab]]
print("out-of-fold AUC | home vs away %.3f | dead vs alive: %s" % (roc_auc_score(v3_h[tm], zt_oof[tm]),
      " | ".join("%s %.3f" % (m, roc_auc_score(v3_d[v3_lab], zd_oof[m][v3_lab])) for m in LIN_MODES)))
print("team weights:", dict(zip(TEAM3_NAMES, np.round(mt_final.coef_.ravel(), 2))))
for m in LIN_MODES:
    print("dead weights [%s]:" % m, dict(zip(DEAD_NAMES[m], np.round(md_final[m].coef_.ravel(), 2))), "| intercept %.2f" % md_final[m].intercept_[0])

out-of-fold AUC | home vs away 0.937 | dead vs alive: relative 0.719 | ball_relative 0.827
team weights: {'learned team model (weak labels)': 1.22, 'nearest-player vote +/-5s': 1.42, 'control evidence +/-3s': 0.14, 'team shape: hull difference +/-2s': -0.0, 'smart-tagging team prior': -0.03}
dead weights [relative]: {'player speed vs local tempo': 1.5, 'slow players vs local tempo': 3.0, 'team spread vs local baseline': -3.07, 'set-piece tag': 1.08} | intercept -1.25
dead weights [ball_relative]: {'player speed vs local tempo': 0.87, 'slow players vs local tempo': 2.65, 'team spread vs local baseline': -3.12, 'set-piece tag': 0.82, 'ball outside pitch': 0.6, 'ball speed (seen frames) +/-3s': -0.44, 'ball distance to the lines': -0.61, 'ball range +/-2.5s (log)': -1.21} | intercept -1.31


### A learned dead-ball model (estimator pinned after a grouped-CV comparison; cross-fitted by minute; calibrated)

The linear dead-ball models see only weighted sums of the cues. A nonlinear estimator can combine them (for example "slow players AND ball far from the lines") if there is signal to find. The estimator family is chosen on the labelled window with **group folds over minutes** (a scikit-learn comparison; the winner, Extra Trees, is pinned in `PIN_ESTIMATORS`), then each minute is predicted by a model that never saw it, and the outputs are Platt-calibrated so they plug into the same decoders. **Keep it only if it clearly beats the linear dead models** in the table below: with about four dead stretches in the window, a flexible model can memorise them and then misjudge the rest of the match, which the whole-match structure checks (and the consistency filter) are there to catch.

In [17]:
Xl, names_l = XD["ball_relative"], DEAD_NAMES["ball_relative"]
sel_rows = v3_lab[::5]; y_l = v3_d[sel_rows].astype(int); g_l = blocks_lab[::5][:len(sel_rows)]

def pooled_oof_auc(est, X_, y_, g_):
    """Leave-one-minute-out predictions pooled over all minutes, then ONE AUC (no per-fold AUC, so a minute without dead frames cannot break it)."""
    oof_ = np.zeros(len(y_))
    for tr_, te_ in LeaveOneGroupOut().split(X_, y_, g_):
        oof_[te_] = clone(est).fit(X_[tr_], y_[tr_]).predict_proba(X_[te_])[:, 1] if len(np.unique(y_[tr_])) > 1 else y_[tr_].mean()
    return roc_auc_score(y_, oof_)

def dead_candidates():
    c = {k: clone(v) for k, v in KNOWN_EST.items()}
    for msl in (5, 10, 20, 40):
        for mf in ("sqrt", 0.5):
            c["ExtraTrees(min_leaf=%d, max_features=%s)" % (msl, mf)] = type(KNOWN_EST["ExtraTreesClassifier"])(n_estimators=200, min_samples_leaf=msl, max_features=mf, random_state=0, n_jobs=1)
    for md_, mi in ((3, 100), (4, 200)):
        c["HistGB(depth=%d, iter=%d)" % (md_, mi)] = HistGradientBoostingClassifier(max_depth=md_, learning_rate=0.05, max_iter=mi, l2_regularization=1.0, random_state=0)
    return c

def pick_dead_estimator():
    pin = PIN_ESTIMATORS.get("dead")
    if pin:
        return make_pinned(pin), "pinned: " + str(pin)
    print("(the comparison pools the leave-one-minute-out predictions and computes one AUC, so a minute without dead frames cannot break it)")
    cand = dead_candidates(); rows_ = [(k, pooled_oof_auc(e, Xl[sel_rows], y_l, g_l)) for k, e in cand.items()]
    board = pd.DataFrame(rows_, columns=["estimator", "pooled out-of-fold AUC"]).sort_values("pooled out-of-fold AUC", ascending=False).reset_index(drop=True)
    print(board.head(10).round(3).to_string(index=False))
    best = board.iloc[0]["estimator"]
    return clone(cand[best]), "pooled-OOF comparison chose " + best

EST_DEAD, EST_NOTE = pick_dead_estimator()
print("dead-ball estimator:", EST_NOTE, "->", type(EST_DEAD).__name__)
print(pin_hint("dead", EST_DEAD))

raw_oof = np.zeros(N); tr_all = v3_lab[::2]
for tr_b, te_b in LeaveOneGroupOut().split(v3_lab, groups=blocks_lab):
    tr_f = v3_lab[tr_b][::2]; te_f = v3_lab[te_b]
    raw_oof[te_f] = clone(EST_DEAD).fit(Xl[tr_f], v3_d[tr_f].astype(int)).predict_proba(Xl[te_f])[:, 1]
platt_d = LogisticRegression(C=1e6).fit(logit(raw_oof[v3_lab]).reshape(-1, 1), v3_d[v3_lab].astype(int))
zd_oof["learned"] = np.zeros(N); zd_oof["learned"][v3_lab] = platt_d.decision_function(logit(raw_oof[v3_lab]).reshape(-1, 1))
est_full = clone(EST_DEAD).fit(Xl[v3_lab[::2]], v3_d[v3_lab[::2]].astype(int))
zd_final["learned"] = platt_d.decision_function(logit(est_full.predict_proba(Xl)[:, 1]).reshape(-1, 1))
print("out-of-fold dead-vs-alive AUC | linear relative %.3f | linear ball_relative %.3f | LEARNED %.3f" % tuple(
    roc_auc_score(v3_d[v3_lab], zd_oof[m][v3_lab]) for m in ("relative", "ball_relative", "learned")))
print("keep the learned model only if it is clearly ahead of the linear ones (about +0.03 AUC or more) AND the structure checks below stay sane")

dead-ball estimator: pinned: ('ExtraTreesClassifier', {'min_samples_leaf': 5, 'n_estimators': 200, 'n_jobs': 1, 'random_state': 0}) -> ExtraTreesClassifier
   to pin it for the final run:  PIN_ESTIMATORS["dead"] = ('ExtraTreesClassifier', {'min_samples_leaf': 5, 'n_estimators': 200, 'n_jobs': 1, 'random_state': 0})
out-of-fold dead-vs-alive AUC | linear relative 0.719 | linear ball_relative 0.827 | LEARNED 0.907
keep the learned model only if it is clearly ahead of the linear ones (about +0.03 AUC or more) AND the structure checks below stay sane


In [18]:
# ---------------- decoder helpers (shared by the team comparison, the grid and the whole-match fit)
seen_all = ball_state > 0

def emission_scores(kind, zt, zd, tau_t, tau_d, seen):
    lt, ld = tau_t * zt, tau_d * zd
    log_d, log_alive = -np.logaddexp(0, -ld), -np.logaddexp(0, ld)
    hier_h, hier_a = log_alive - np.logaddexp(0, -lt), log_alive - np.logaddexp(0, lt)          # alive = log p(alive) + log p(team | alive)
    bal_h, bal_a = log_alive + np.minimum(0.0, lt), log_alive - np.maximum(0.0, lt)           # only the LESS likely team is penalised
    if kind == "hierarchical": h, a = hier_h, hier_a
    elif kind == "balanced": h, a = bal_h, bal_a
    else: h, a = np.where(seen, hier_h, bal_h), np.where(seen, hier_a, bal_a)                  # mixed: hierarchical where the ball is seen
    return np.stack([h, a, log_d], axis=1)

def _post(path):
    return enforce_min_duration(path, int(round(CFG["min_possession_s"] * FPS)), int(round(CFG["min_dead_s"] * FPS)))

def decode_match_final(kind, zt, zd, tt, td, sw, dc):
    sc = emission_scores(kind, zt, zd, tt, td, seen_all); lab_ = np.full(N, "X", dtype="<U1")
    for half, a_, b_ in HALVES:
        s_ = sc[a_:b_ + 1]; kick = MATCH["pre_kickoff_dead"].get(half); kk = (kick - a_) if kick else 0
        lab_[a_:b_ + 1] = LETTER[np.r_[np.full(kk, 2, np.int8), _post(viterbi(s_[kk:], sw, dc))]]
    return lab_
def decoded_share(lab_):
    return ((lab_ == "D") & GRADED).sum() / GRADED.sum()
def fit_delta(kind, zt, zd, p, target, iters=8):
    lo, hi = -6.0, 6.0
    for _ in range(iters):
        mid = (lo + hi) / 2
        if decoded_share(decode_match_final(kind, zt, zd + mid, *p)) > target: hi = mid
        else: lo = mid
    return (lo + hi) / 2

def home_share_alive(lab_):
    h, a_ = ((lab_ == "H") & GRADED).sum(), ((lab_ == "A") & GRADED).sum()
    return h / max(h + a_, 1)


### Team-model comparison

Each home/away model below is scored on the labelled window (out-of-fold AUC and accuracy on the truly home/away frames, leave-one-minute-out). The best AUC is used; `FINAL_TEAM_MODEL` overrides the choice.

* **A** weak-label model alone (the cross-fitted LDA from Step 5b, no window fit)
* **B** a logistic stack of the weak-label model, the nearest-player vote, control evidence, team-shape difference and the tag prior
* **C** B without the Smart Tagging prior, **D** B without the team-shape difference, **E** B without the weak-label model
* **F** the stack with a nonlinear learner (ExtraTrees, home/away-symmetrised)

In [19]:
from sklearn.ensemble import ExtraTreesClassifier as _ET
def lr_team(cols):
    X_ = XT3[:, cols]; zo = np.zeros(N)
    def fit(frames):
        f = frames[(v3_h | v3_a)[frames]]
        return LogisticRegression(C=FINAL_C, fit_intercept=False, max_iter=3000).fit(X_[f], v3_h[f].astype(int))
    for tr_b, te_b in LeaveOneGroupOut().split(v3_lab, groups=blocks_lab):
        zo[v3_lab[te_b]] = X_[v3_lab[te_b]] @ fit(v3_lab[tr_b]).coef_.ravel()
    return zo, X_ @ fit(v3_lab).coef_.ravel()
def nonlinear_team():
    zo = np.zeros(N)
    def fit(frames):
        f = frames[(v3_h | v3_a)[frames]]; Xa = np.vstack([XT3[f], -XT3[f]]); ya = np.r_[v3_h[f].astype(int), 1 - v3_h[f].astype(int)]   # every column is antisymmetric: swapping teams = negating
        return _ET(n_estimators=200, min_samples_leaf=20, random_state=0, n_jobs=1).fit(Xa, ya)
    def logit_sym(m, X_):
        p = 0.5 * (m.predict_proba(X_)[:, 1] + 1 - m.predict_proba(-X_)[:, 1]); return np.log(np.clip(p, 1e-4, 1 - 1e-4) / np.clip(1 - p, 1e-4, 1))
    for tr_b, te_b in LeaveOneGroupOut().split(v3_lab, groups=blocks_lab):
        zo[v3_lab[te_b]] = logit_sym(fit(v3_lab[tr_b]), XT3[v3_lab[te_b]])
    return zo, logit_sym(fit(v3_lab), XT3)

zw = EMIS["5b"]["ml_team"].astype(float)
TEAM_VARIANTS = {"A weak-label model only": (zw, zw), "B stack (current)": lr_team([0, 1, 2, 3, 4]), "C stack without tag prior": lr_team([0, 1, 2, 3]),
                 "D stack without team shape": lr_team([0, 1, 2, 4]), "E stack without weak-label model": lr_team([1, 2, 3, 4]), "F stack, nonlinear (ExtraTrees)": nonlinear_team()}
tm_ = v3_lab[(v3_h | v3_a)[v3_lab]]; rows_ = []
for name, (zo, zf) in TEAM_VARIANTS.items():
    rows_.append(dict(variant=name, window_AUC=roc_auc_score(v3_h[tm_], zo[tm_]), window_team_acc=((zo[tm_] > 0) == v3_h[tm_]).mean()))
TEAMCMP = pd.DataFrame(rows_).set_index("variant"); TEAMCMP.index = [i[0] + " " + i[2:] for i in TEAMCMP.index]
print("TEAM MODELS: out-of-fold on the labelled frames of the window (leave-one-minute-out); the best AUC is used")
print(TEAMCMP.round(3).to_string())
pick_letter = TEAMCMP["window_AUC"].idxmax()[0] if FINAL_TEAM_MODEL == "auto" else FINAL_TEAM_MODEL
chosen = [k for k in TEAM_VARIANTS if k[0] == pick_letter][0]
zt_oof, zt_final = TEAM_VARIANTS[chosen][0].copy(), TEAM_VARIANTS[chosen][1].copy()
print("\nTEAM MODEL USED:", chosen)

TEAM MODELS: out-of-fold on the labelled frames of the window (leave-one-minute-out); the best AUC is used
                                  window_AUC  window_team_acc
A weak-label model only                0.944            0.867
B stack (current)                      0.937            0.876
C stack without tag prior              0.949            0.876
D stack without team shape             0.938            0.875
E stack without weak-label model       0.934            0.871
F stack, nonlinear (ExtraTrees)        0.951            0.859

TEAM MODEL USED: F stack, nonlinear (ExtraTrees)


In [20]:
# ---------------- emission designs, window grid with a dead-offset, NESTED (selection-honest) estimate
# (emission_scores, _post, seen_all come from the decoder-helper cell above)
run_bounds = np.cumsum([0] + [hi - lo for lo, hi in runs_v3]); seen_lab = seen_all[v3_lab]
def decode_lab(kind, ztl, zdl, tt, td, sw, dc, off):
    sc = emission_scores(kind, ztl, zdl + off, tt, td, seen_lab); pred = np.empty(len(ztl), np.int8)
    for i in range(len(runs_v3)):
        pred[run_bounds[i]:run_bounds[i + 1]] = _post(viterbi(sc[run_bounds[i]:run_bounds[i + 1]], sw, dc))
    return pred
def block_cm(pred):
    return np.bincount(blocks_lab * 9 + tc_lab * 3 + pred, minlength=nb_ * 9).reshape(nb_, 3, 3).astype(float)
def metrics_from_cm(cm):
    tp = np.diag(cm); rec = tp / np.maximum(cm.sum(1), 1); prec = tp / np.maximum(cm.sum(0), 1); f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return dict(Accuracy=tp.sum() / max(cm.sum(), 1), Precision=prec.mean(), Recall=rec.mean(), F1=f1.mean())
def to_full(pred):
    lab_ = np.full(N, "X", dtype="<U1"); lab_[v3_lab] = LETTER[pred]; return lab_

keys = list(product(FINAL_EMISSIONS, FINAL_DEAD_MODES)); CAND = {}
ztl = zt_oof[v3_lab]; t0 = time.time()
settings = list(product(FINAL_GRID["tau_t"], FINAL_GRID["tau_d"], FINAL_GRID["switch"], FINAL_GRID["dead_cost"], FINAL_GRID["offset"]))
for kind, dm in keys:
    zdl = zd_oof[dm][v3_lab]; cms = np.array([block_cm(decode_lab(kind, ztl, zdl, *s)) for s in settings])         # (settings, blocks, 3, 3)
    f1s = np.array([metrics_from_cm(cms[i].sum(0))["F1"] for i in range(len(settings))])
    best_i, best_key, best_fm = None, None, None
    for i in np.argsort(-f1s)[:FINAL_TOPK]:                                      # stage 2: among the best-F1 settings, the best spellF1 (the ranking metric)
        fm = frame_metrics(to_full(decode_lab(kind, ztl, zdl, *settings[i]))); k_ = (round(fm["spellF1"], 4), float(f1s[i]))
        if best_key is None or k_ > best_key: best_i, best_key, best_fm = int(i), k_, fm
    tot = np.zeros((3, 3))
    for b in range(nb_):                                                         # nested F1: choose on the other blocks, score the held-out one
        others = cms.sum(1) - cms[:, b]; best = int(np.argmax([metrics_from_cm(o)["F1"] for o in others])); tot += cms[best, b]
    CAND[(kind, dm)] = dict(setting=settings[best_i], plain=metrics_from_cm(cms[best_i].sum(0)), nested=metrics_from_cm(tot), spellF1=best_fm["spellF1"], possessions=best_fm["possessions"])
print("%d candidates x %d settings in %.0fs\n" % (len(keys), len(settings), time.time() - t0))
tab = pd.DataFrame({"%s + %s" % k: {**{"plain " + m: v["plain"][m] for m in ("Accuracy", "Precision", "Recall", "F1")}, "plain spellF1": v["spellF1"], "possessions": v["possessions"],
                                    **{"NESTED " + m: v["nested"][m] for m in ("Accuracy", "Precision", "Recall", "F1")}} for k, v in CAND.items()}).T
print("plain = the chosen setting on the same window (optimistic; chosen for the best spellF1 among the %d best-F1 settings) | NESTED = setting chosen on four minutes by F1, scored on the fifth (honest frame metrics)" % FINAL_TOPK)
print(tab.round(3).to_string())
for k, v in CAND.items():
    s = v["setting"]; print("  %-28s best: tau_t %.2f tau_d %.2f switch %g dead_cost %g offset %+.1f" % ("%s + %s" % k, *s))
print("\ntruth possessions in the window:", frame_metrics(to_full(np.minimum(tc_lab, 2).astype(np.int8)))["truth_possessions"])

9 candidates x 648 settings in 234s

plain = the chosen setting on the same window (optimistic; chosen for the best spellF1 among the 30 best-F1 settings) | NESTED = setting chosen on four minutes by F1, scored on the fifth (honest frame metrics)
                              plain Accuracy  plain Precision  plain Recall  plain F1  plain spellF1  possessions  NESTED Accuracy  NESTED Precision  NESTED Recall  NESTED F1
hierarchical + relative                0.822            0.830         0.800     0.811          0.417          7.0            0.776             0.770          0.730      0.743
hierarchical + ball_relative           0.816            0.820         0.791     0.801          0.417          7.0            0.777             0.788          0.734      0.752
hierarchical + learned                 0.805            0.801         0.781     0.789          0.432          8.0            0.763             0.747          0.743      0.740
balanced + relative                    0.787         

In [21]:
# ---------------- whole match: fit the dead share on the DECODED output, test the structure, choose the final candidate
def structure_stats(lab_):
    d_ = (lab_ == "D") & GRADED; ds_, dl_ = run_lengths(d_); gm = GRADED.sum() / FPS / 60
    st = pm.possession_stats(pm.label_codes(lab_), [(a_, b_ + 1) for _, a_, b_ in HALVES]); miss, seen = GRADED & ~ball_obs, GRADED & ball_obs
    return dict(dead_share=d_.sum() / GRADED.sum(), dead_if_missing=d_[miss].sum() / max(miss.sum(), 1), dead_if_seen=d_[seen].sum() / max(seen.sum(), 1),
                dead_stretches=len(dl_), dead_stretches_per_min=len(dl_) / gm, dead_mean_s=dl_.mean() / FPS if len(dl_) else 0.0, possessions=st["n_spells"], poss_per_min=st["n_spells"] / gm, poss_mean_s=st["mean_duration_s"])
def is_sane(s):
    S = FINAL_SANITY
    return bool(S["dead_stretches_per_min"][0] <= s["dead_stretches_per_min"] <= S["dead_stretches_per_min"][1] and S["dead_mean_s"][0] <= s["dead_mean_s"] <= S["dead_mean_s"][1]
                and S["poss_per_min"][0] <= s["poss_per_min"] <= S["poss_per_min"][1] and S["poss_mean_s"][0] <= s["poss_mean_s"] <= S["poss_mean_s"][1])

t0 = time.time(); rows, LAB_CAND, DELTA = [], {}, {}
for (kind, dm), v in CAND.items():
    p = v["setting"][:4]; d_ = fit_delta(kind, zt_final, zd_final[dm], p, FINAL_DEAD_TARGET); DELTA[(kind, dm)] = d_
    LAB_CAND[(kind, dm)] = decode_match_final(kind, zt_final, zd_final[dm] + d_, *p); s = structure_stats(LAB_CAND[(kind, dm)])
    rows.append(dict(candidate="%s + %s" % (kind, dm), nested_acc=v["nested"]["Accuracy"], nested_F1=v["nested"]["F1"], window_offset=v["setting"][4], match_delta=d_,
                     **{k: s[k] for k in ("dead_share", "dead_if_missing", "dead_if_seen", "dead_stretches", "dead_stretches_per_min", "dead_mean_s", "poss_per_min", "poss_mean_s")}, sane=is_sane(s)))
WM = pd.DataFrame(rows).set_index("candidate")
print("WHOLE-MATCH structure with the decoded dead share fitted to %.2f (%.0fs):" % (FINAL_DEAD_TARGET, time.time() - t0))
print(WM.round(2).to_string())
print("\nwindow_offset = the dead operating point that is best on the labelled window; match_delta = the shift that gives the assumed whole-match dead share.")
print("A prior shift from a 20% to a 38% dead share is about +0.9 in log-odds: candidates whose two values disagree strongly are mis-specified for the match.")

WM["spellF1"] = [CAND[tuple(i.split(" + "))]["spellF1"] for i in WM.index]
WM["consistent"] = (WM["match_delta"] - WM["window_offset"]).abs() <= FINAL_CONSISTENCY
FRAME_COL = {"Accuracy": "nested_acc", "F1": "nested_F1"}[FINAL_FRAME_METRIC]
WM["score"] = FINAL_WEIGHT_SPELL * WM["spellF1"] + (1 - FINAL_WEIGHT_SPELL) * WM[FRAME_COL]
print("\nCANDIDATES: score = %.1f * spellF1 + %.1f * nested %s; consistent = |match shift - window offset| <= %.1f" % (FINAL_WEIGHT_SPELL, 1 - FINAL_WEIGHT_SPELL, FINAL_FRAME_METRIC, FINAL_CONSISTENCY))
print(WM[["nested_acc", "nested_F1", "spellF1", "score", "match_delta", "window_offset", "consistent", "sane"]].round(3).to_string())
ok = WM[WM["sane"] & WM["consistent"]]
if len(ok):
    FINAL_KEY = tuple(ok["score"].idxmax().split(" + ")); why = "highest score (spellF1 / nested frame-metric blend) among the %d candidates that are structurally sane AND consistent between window and match" % len(ok)
else:
    ok2 = WM[WM["sane"]]
    if len(ok2):
        FINAL_KEY = tuple(ok2["score"].idxmax().split(" + ")); why = "NO candidate was both sane and consistent: took the best score among the %d sane ones (CHECK the consistency column)" % len(ok2)
    else:
        FINAL_KEY = FINAL_FALLBACK; why = "NO candidate passed the structure checks: fell back to %s" % (" + ".join(FINAL_FALLBACK),)
print("\nFINAL CHOICE: %s  | reason: %s" % (" + ".join(FINAL_KEY), why))

WHOLE-MATCH structure with the decoded dead share fitted to 0.38 (59s):
                              nested_acc  nested_F1  window_offset  match_delta  dead_share  dead_if_missing  dead_if_seen  dead_stretches  dead_stretches_per_min  dead_mean_s  poss_per_min  poss_mean_s  sane
candidate                                                                                                                                                                                                       
hierarchical + relative             0.78       0.74           -0.5        -2.32        0.38             0.67          0.18              64                    0.60        37.81          1.29        28.72  True
hierarchical + ball_relative        0.78       0.75            0.0        -3.02        0.38             0.74          0.14              67                    0.63        36.16          1.37        27.11  True
hierarchical + learned              0.76       0.74           -1.0        -2.32        0.38 

In [22]:
# ---------------- the final labels, report, record
kind, dm = FINAL_KEY; v = CAND[FINAL_KEY]; p = v["setting"][:4]; d_fin = DELTA[FINAL_KEY]
labels = LAB_CAND[FINAL_KEY].copy()
print("FINAL MODEL: emission '%s' | dead cues '%s' | team model '%s' | tau_t %.2f tau_d %.2f switch %g dead_cost %g | dead shift %+.2f\n" % ((kind, dm, chosen) + tuple(p) + (d_fin,)))
print("home share of the alive frames in the final labels: %.1f%% (an outcome; nothing is fitted to it)\n" % (100 * home_share_alive(labels)))
pred = decode_lab(kind, ztl, zd_oof[dm][v3_lab], *v["setting"]); cm = block_cm(pred).sum(0)
tp = np.diag(cm); rec = tp / np.maximum(cm.sum(1), 1); prec = tp / np.maximum(cm.sum(0), 1); f1c = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
print("OUT-OF-FOLD per class on the labelled window:"); print(pd.DataFrame({"precision": prec, "recall": rec, "F1": f1c}, index=["home", "away", "dead"]).round(3).to_string())
print("\nNESTED (honest) window metrics:", {k: round(x, 3) for k, x in v["nested"].items()}, "| plain spellF1 %.3f | possessions %d vs %d true" % (v["spellF1"], v["possessions"], frame_metrics(to_full(np.minimum(tc_lab, 2).astype(np.int8)))["truth_possessions"]))
print("\nsubmitted labels scored on the labelled window (whole-match decode WITH the dead shift; the weights saw these frames, so read it as a sanity check):")
headline(evaluate_labels(labels), "final labels")
print("\nstructure of the submitted labels:", {k: round(float(x), 2) for k, x in structure_stats(labels).items()})
print("label counts on graded frames:", {l: int(((labels == l) & GRADED).sum()) for l in "HAD"})
(ROOT / "runs").mkdir(exist_ok=True)
(ROOT / "runs" / (TEAM_NAME + ("_synthetic" if SYNTHETIC else "") + "_final_model.json")).write_text(json.dumps(dict(
    emission=kind, dead_cues=dm, setting=dict(zip(["tau_t", "tau_d", "switch", "dead_cost", "window_offset"], map(float, v["setting"]))), match_delta=float(DELTA[FINAL_KEY]),
    dead_target=FINAL_DEAD_TARGET, dead_shift_final=float(d_fin), team_model=chosen, home_share_final=float(home_share_alive(labels)), nested={k: float(x) for k, x in v["nested"].items()}, reason=why,
    pin_estimators={"team": pin_hint("team", SELECTED["team"]).split(" = ", 1)[1] if SELECTED.get("team") is not None else None, "dead": pin_hint("dead", EST_DEAD).split(" = ", 1)[1]}), indent=1))

FINAL MODEL: emission 'mixed' | dead cues 'relative' | team model 'F stack, nonlinear (ExtraTrees)' | tau_t 0.20 tau_d 0.40 switch 8 dead_cost 30 | dead shift +0.21

home share of the alive frames in the final labels: 63.8% (an outcome; nothing is fitted to it)

OUT-OF-FOLD per class on the labelled window:
      precision  recall     F1
home      0.778   0.904  0.836
away      0.799   0.794  0.796
dead      0.813   0.511  0.627

NESTED (honest) window metrics: {'Accuracy': 0.752, 'Precision': 0.725, 'Recall': 0.697, 'F1': 0.705} | plain spellF1 0.350 | possessions 7 vs 17 true

submitted labels scored on the labelled window (whole-match decode WITH the dead shift; the weights saw these frames, so read it as a sanity check):
final labels                 spellF1 0.5000 | macroF1 0.8818 | alive/dead F1 0.953 | team acc (GT alive) 0.895 | possessions pred 7 vs GT 17 | mean dur 33.2s vs 14.2s

structure of the submitted labels: {'dead_share': 0.38, 'dead_if_missing': 0.58, 'dead_if_seen': 

816

## Write, validate and checksum the submission

In [23]:
def write_submission(labels, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"frame": np.arange(len(labels)), "label": labels}).to_csv(path, index=False)
    return path

out_path = (ROOT / "cache" / "synthetic" / (TEAM_NAME + "_synthetic.csv")) if SYNTHETIC \
    else (ROOT / "evaluation" / "results" / (TEAM_NAME + ".csv"))
write_submission(labels, out_path)

table = pm.read_label_csv(out_path)
problem = pm.validate_submission(table, N)
print("validation:", "OK" if problem is None else "FAILED -> " + problem)
print("rows:", len(table), "| labels:", table["label"].value_counts().to_dict())

md5 = hashlib.md5(Path(out_path).read_bytes()).hexdigest()
runs_dir = ROOT / "runs"; runs_dir.mkdir(exist_ok=True)
record = dict(team=TEAM_NAME, mode="synthetic" if SYNTHETIC else "real", written=time.strftime("%Y-%m-%d %H:%M:%S"),
              csv=str(out_path.relative_to(ROOT)), md5=md5, numpy=np.__version__, pandas=pd.__version__, config=CFG)
cfg_path = runs_dir / (TEAM_NAME + ("_synthetic" if SYNTHETIC else "") + "_config.json")
cfg_path.write_text(json.dumps(record, indent=1, default=list))
print("config + checksum saved:", cfg_path.relative_to(ROOT), "| md5", md5)

# ---- the same run record as a flat CSV (key,value) next to the JSON files, and the full path of everything exported
def _flatten(d, prefix=""):
    out = {}
    for k, x in d.items():
        key = prefix + str(k)
        if isinstance(x, dict): out.update(_flatten(x, key + "."))
        elif isinstance(x, (list, tuple)): out[key] = json.dumps(list(x), default=str)
        else: out[key] = x
    return out
final_json = runs_dir / (TEAM_NAME + ("_synthetic" if SYNTHETIC else "") + "_final_model.json")
flat = _flatten({k: x for k, x in record.items() if k != "written"}, "run.")          # no timestamp: the summary is reproducible too
if final_json.exists():
    flat.update(_flatten(json.loads(final_json.read_text()), "final_model."))
summary_path = runs_dir / (TEAM_NAME + ("_synthetic" if SYNTHETIC else "") + "_run_summary.csv")
pd.DataFrame({"key": list(flat), "value": list(flat.values())}).to_csv(summary_path, index=False)
print("run summary (csv):", summary_path.relative_to(ROOT))
print("\nexported files (full paths):")
for f_ in (out_path, cfg_path, final_json, summary_path):
    if Path(f_).exists():
        print("  %s   (%d bytes)" % (Path(f_).resolve(), Path(f_).stat().st_size))

# Keep derived match data out of git (the terms forbid publishing it).
gi = ROOT / ".gitignore"
text = gi.read_text() if gi.exists() else ""
missing = [e for e in ("cache/", "data/") if e not in text]
if missing:
    gi.write_text(text + "\n# derived / gated match data - never commit\n" + "\n".join(missing) + "\n")
    print("added to .gitignore:", missing)

validation: OK
rows: 199319 | labels: {'H': 62982, 'D': 60100, 'X': 40462, 'A': 35775}
config + checksum saved: runs/Cules of Duty_config.json | md5 a62d4c09ce5107ad97c9be15b86e8a42
run summary (csv): runs/Cules of Duty_run_summary.csv

exported files (full paths):
  /Users/patrickkingori/Downloads/more-than-a-hack-2027-main/evaluation/results/Cules of Duty.csv   (1682773 bytes)
  /Users/patrickkingori/Downloads/more-than-a-hack-2027-main/runs/Cules of Duty_config.json   (1417 bytes)
  /Users/patrickkingori/Downloads/more-than-a-hack-2027-main/runs/Cules of Duty_final_model.json   (816 bytes)
  /Users/patrickkingori/Downloads/more-than-a-hack-2027-main/runs/Cules of Duty_run_summary.csv   (2523 bytes)


### Final run and deliverables checklist (the competition checks all three)

1. **Freeze the choices in the notebook:** `TEAM_NAME = "Cules of Duty"`, `FORCE_SYNTHETIC = False`, and the two estimator pins in `PIN_ESTIMATORS` (already set). The final configuration is chosen by the last cell of the final-model step (`FINAL_KEY`); whatever sets `labels` last is what Step 9 writes.
2. **Regenerate from scratch without Jupyter:** `python reproduce.py` from the repository root. It prints the CSV path and its **MD5**. Run it **twice**: the two MD5 values must match, because the organisers will rerun your repository and expect *the same file*.
3. **Validate the exact file you will upload:** `python evaluation/validate_submission.py "evaluation/results/Cules of Duty.csv"` (keep the quotes: the name contains spaces).
4. **Pin the environment:** the next cell writes `requirements-pinned.txt` from the versions installed here. Commit it and state the Python version.
5. **Repository hygiene:** `data/` and `cache/` must not be committed (the match data is not redistributable and the repository must be public). Check with `git status` and `git ls-files | grep -i -E "data/|cache/"` before pushing.
6. **Technical document:** fill the template (approach, assets used *and dropped*, possession rules, limitations, reproduction), name it `Cules of Duty.pdf` or `Cules of Duty.md`.
7. **Upload** through the Google Form (needs a Google account): team name (`Cules of Duty`), `Cules of Duty.csv`, the repository URL, the document. Deadline: **November 17**. One response per team; the last one before the deadline counts.

In [24]:
import sys, platform
from importlib import metadata
# Pin only what this run actually imported (a package that was merely installed is not a dependency of the pipeline).
MODULES = {"numpy": ("numpy",), "pandas": ("pandas",), "matplotlib": ("matplotlib",), "scipy": ("scipy",), "sklearn": ("scikit-learn",),
           "imblearn": ("imbalanced-learn",), "cv2": ("opencv-python-headless", "opencv-python"), "kloppy": ("kloppy",)}
rows = []
for mod, dists in MODULES.items():
    if mod not in sys.modules:
        continue
    for dist in dists:
        try:
            rows.append("%s==%s" % (dist, metadata.version(dist))); break
        except metadata.PackageNotFoundError:
            continue
(ROOT / "requirements-pinned.txt").write_text("# python %s on %s\n%s\n" % (platform.python_version(), platform.platform(), "\n".join(rows)))
print("wrote requirements-pinned.txt:"); print("  python", platform.python_version()); print("  " + "\n  ".join(rows))
print("\nReproduce with:  pip install -r requirements-pinned.txt   then   python reproduce.py --notebook possession_pipeline_FINAL_v4.ipynb")

wrote requirements-pinned.txt:
  python 3.11.18
  numpy==1.26.4
  pandas==2.1.4
  matplotlib==3.7.5
  scipy==1.11.4
  scikit-learn==1.4.2

Reproduce with:  pip install -r requirements-pinned.txt   then   python reproduce.py --notebook possession_pipeline_FINAL_v4.ipynb
